In [1]:
import numpy as np

from pathlib import Path



base = Path(r"C:\Users\Ramnath Lakshmanan\Downloads\Initial_data_points_starter (2)\initial_data")



X1 = np.load(base / "function_1" / "initial_inputs.npy")

y1 = np.load(base / "function_1" / "initial_outputs.npy")



print("X shape:", X1.shape)

print("y shape:", y1.shape)

print(X1)

print(y1)

X shape: (10, 2)
y shape: (10,)
[[0.31940389 0.76295937]
 [0.57432921 0.8798981 ]
 [0.73102363 0.73299988]
 [0.84035342 0.26473161]
 [0.65011406 0.68152635]
 [0.41043714 0.1475543 ]
 [0.31269116 0.07872278]
 [0.68341817 0.86105746]
 [0.08250725 0.40348751]
 [0.88388983 0.58225397]]
[ 1.32267704e-079  1.03307824e-046  7.71087511e-016  3.34177101e-124
 -3.60606264e-003 -2.15924904e-054 -2.08909327e-091  2.53500115e-040
  3.60677119e-081  6.22985647e-048]


In [5]:
import numpy as np

from pathlib import Path



base = Path(r"C:\Users\Ramnath Lakshmanan\Downloads\Initial_data_points_starter (2)\initial_data")



data = {}

for k in range(1, 9):

    folder = base / f"function_{k}"

    X = np.load(folder / "initial_inputs.npy")

    y = np.load(folder / "initial_outputs.npy").ravel()

    data[k] = {"X": X, "y": y}

    print(f"F{k}: X={X.shape}, y={y.shape}, y_min={y.min():.4g}, y_max={y.max():.4g}")

F1: X=(10, 2), y=(10,), y_min=-0.003606, y_max=7.711e-16
F2: X=(10, 2), y=(10,), y_min=-0.06562, y_max=0.6112
F3: X=(15, 3), y=(15,), y_min=-0.3989, y_max=-0.03484
F4: X=(30, 4), y=(30,), y_min=-32.63, y_max=-4.026
F5: X=(20, 4), y=(20,), y_min=0.1129, y_max=1089
F6: X=(20, 5), y=(20,), y_min=-2.571, y_max=-0.7143
F7: X=(30, 6), y=(30,), y_min=0.002701, y_max=1.365
F8: X=(40, 8), y=(40,), y_min=5.592, y_max=9.598


In [8]:
import numpy as np

from pathlib import Path

from sklearn.gaussian_process import GaussianProcessRegressor

from sklearn.gaussian_process.kernels import ConstantKernel, Matern, WhiteKernel

from sklearn.preprocessing import StandardScaler



base = Path(r"C:\Users\Ramnath Lakshmanan\Downloads\Initial_data_points_starter (2)\initial_data")



DIMS = {1: 2, 2: 2, 3: 3, 4: 4, 5: 4, 6: 5, 7: 6, 8: 8}



POLICY = {

    1: dict(kappa=2.6, n_cand=8000, min_dist=0.05),

    2: dict(kappa=2.4, n_cand=8000, min_dist=0.05),

    3: dict(kappa=2.2, n_cand=8000, min_dist=0.08),

    4: dict(kappa=2.0, n_cand=10000, min_dist=0.10),

    5: dict(kappa=1.8, n_cand=10000, min_dist=0.10),

    6: dict(kappa=1.8, n_cand=12000, min_dist=0.12),

    7: dict(kappa=1.7, n_cand=12000, min_dist=0.12),

    8: dict(kappa=1.6, n_cand=15000, min_dist=0.15),

}



def format_query(x):

    x = np.clip(np.asarray(x, dtype=float).ravel(), 0.0, 1.0)

    return "-".join(f"{v:.6f}" for v in x)



def lhs(n, d, rng):

    u = np.empty((n, d))

    edges = np.linspace(0.0, 1.0, n + 1)

    low, high = edges[:-1], edges[1:]

    for j in range(d):

        u[:, j] = rng.uniform(low, high)

        rng.shuffle(u[:, j])

    return u



queries = {}

for k in range(1, 9):

    X = np.load(base / f"function_{k}" / "initial_inputs.npy").astype(float)

    y = np.load(base / f"function_{k}" / "initial_outputs.npy").astype(float).ravel()

    d = DIMS[k]

    pol = POLICY[k]

    rng = np.random.default_rng(1000 + k)



    print(f"Fitting F{k} ...")

    xsc = StandardScaler().fit(X)

    kernel = (

        ConstantKernel(1.0, (1e-3, 1e3))

        * Matern(length_scale=np.full(d, 0.3), length_scale_bounds=(0.05, 2.0), nu=2.5)

        + WhiteKernel(1e-5, (1e-10, 1e-1))

    )

    gp = GaussianProcessRegressor(

        kernel=kernel,

        normalize_y=True,

        n_restarts_optimizer=2,

        random_state=k,

        alpha=1e-8,

    )

    gp.fit(xsc.transform(X), y)



    n_half = pol["n_cand"] // 2

    cand = np.vstack([

        lhs(n_half, d, rng),

        rng.uniform(0.0, 1.0, size=(n_half, d)),

    ])

    mu, std = gp.predict(xsc.transform(cand), return_std=True)

    scores = mu + pol["kappa"] * std



    dist = np.min(np.linalg.norm(cand[:, None, :] - X[None, :, :], axis=2), axis=1)

    scores = np.where(dist < pol["min_dist"], -np.inf, scores)



    idx = int(np.argmax(scores))

    q = format_query(cand[idx])

    queries[k] = q

    best_i = int(np.argmax(y))

    print(f"F{k}  best_y={y[best_i]:.6g}  at {format_query(X[best_i])}")

    print(f"     pred_mu={mu[idx]:.4g}  pred_std={std[idx]:.4g}")

    print(f"     SUBMIT: {q}")

    print()



print("=" * 60)

print("PASTE INTO THE PORTAL")

print("=" * 60)

for k, q in queries.items():

    print(f"Function {k}: {q}")

Fitting F1 ...
F1  best_y=7.71088e-16  at 0.731024-0.733000
     pred_mu=-0.0002487  pred_std=0.001064
     SUBMIT: 0.973822-0.110674

Fitting F2 ...
F2  best_y=0.611205  at 0.702637-0.926564
     pred_mu=0.5264  pred_std=0.1593
     SUBMIT: 0.711216-0.383626

Fitting F3 ...


C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


F3  best_y=-0.0348353  at 0.492581-0.611593-0.340176
     pred_mu=-0.05273  pred_std=0.07938
     SUBMIT: 0.628000-0.359224-0.539509

Fitting F4 ...
F4  best_y=-4.02554  at 0.577766-0.428772-0.425826-0.249007
     pred_mu=-2.428  pred_std=2.353
     SUBMIT: 0.367191-0.464182-0.391040-0.448820

Fitting F5 ...
F5  best_y=1088.86  at 0.224189-0.846480-0.879484-0.878516
     pred_mu=1105  pred_std=90.42
     SUBMIT: 0.245752-0.932264-0.962162-0.965058

Fitting F6 ...


C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 2 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 3 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
C:\Users\R

F6  best_y=-0.714265  at 0.728186-0.154693-0.732552-0.693997-0.056401
     pred_mu=-0.6752  pred_std=0.3216
     SUBMIT: 0.446820-0.321047-0.436584-0.762321-0.003307

Fitting F7 ...


C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 2 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


F7  best_y=1.36497  at 0.057896-0.491672-0.247422-0.218118-0.420428-0.730970
     pred_mu=1.225  pred_std=0.1475
     SUBMIT: 0.086332-0.371883-0.229157-0.106524-0.394874-0.689739

Fitting F8 ...
F8  best_y=9.59848  at 0.056447-0.065956-0.022929-0.038786-0.403935-0.801055-0.488307-0.893085
     pred_mu=9.622  pred_std=0.5586
     SUBMIT: 0.054501-0.081016-0.320114-0.398440-0.621808-0.649205-0.338643-0.601487

PASTE INTO THE PORTAL
Function 1: 0.973822-0.110674
Function 2: 0.711216-0.383626
Function 3: 0.628000-0.359224-0.539509
Function 4: 0.367191-0.464182-0.391040-0.448820
Function 5: 0.245752-0.932264-0.962162-0.965058
Function 6: 0.446820-0.321047-0.436584-0.762321-0.003307
Function 7: 0.086332-0.371883-0.229157-0.106524-0.394874-0.689739
Function 8: 0.054501-0.081016-0.320114-0.398440-0.621808-0.649205-0.338643-0.601487


C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 3 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 4 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
C:\Users\R

In [9]:
import numpy as np



raw = " 0.628000-0.359224-0.539509"   # e.g. "0.012345-1.000000-0.500000"



def fix_query(s, dim):

    parts = [p.strip() for p in s.replace(",", "-").split("-") if p.strip()]

    x = np.array([float(p) for p in parts], dtype=float)

    if len(x) != dim:

        raise ValueError(f"expected {dim} numbers, got {len(x)}: {s}")

    # keep every coordinate inside (0, 1) so it always starts with 0.

    x = np.clip(x, 1e-6, 1.0 - 1e-6)

    return "-".join(f"{v:.6f}" for v in x)



print(fix_query(raw, 3))

0.628000-0.359224-0.539509


In [1]:
import numpy as np

from pathlib import Path

from sklearn.gaussian_process import GaussianProcessRegressor

from sklearn.gaussian_process.kernels import ConstantKernel, Matern, WhiteKernel

from sklearn.preprocessing import StandardScaler



base = Path(r"C:\Users\Ramnath Lakshmanan\Downloads\Initial_data_points_starter (2)\initial_data")



DIMS = {1: 2, 2: 2, 3: 3, 4: 4, 5: 4, 6: 5, 7: 6, 8: 8}



WEEK1_X = {

    1: np.array([0.973822, 0.110674]),

    2: np.array([0.711216, 0.383626]),

    3: np.array([0.628013, 0.359224, 0.539509]),

    4: np.array([0.367191, 0.464182, 0.391040, 0.448820]),

    5: np.array([0.245752, 0.932264, 0.962162, 0.965058]),

    6: np.array([0.446820, 0.321047, 0.436584, 0.762321, 0.003307]),

    7: np.array([0.086332, 0.371883, 0.229157, 0.106524, 0.394874, 0.689739]),

    8: np.array([0.054501, 0.081016, 0.320114, 0.398440, 0.621808, 0.649205, 0.338643, 0.601487]),

}

WEEK1_Y = {

    1: -1.1814831817160104e-270,

    2: 0.6525848308450871,

    3: -0.01927280144331694,

    4: -0.9174251874710708,

    5: 2830.5996301272953,

    6: -0.41995556157505487,

    7: 1.5145555220314069,

    8: 9.7443661466821,

}

POLICY = {

    1: dict(kappa=2.6, n_cand=8000, min_dist=0.06, local=0.00, local_frac=0.00),

    2: dict(kappa=1.2, n_cand=8000, min_dist=0.02, local=0.12, local_frac=0.55),

    3: dict(kappa=1.3, n_cand=8000, min_dist=0.03, local=0.14, local_frac=0.55),

    4: dict(kappa=1.2, n_cand=10000, min_dist=0.04, local=0.12, local_frac=0.60),

    5: dict(kappa=1.0, n_cand=10000, min_dist=0.04, local=0.10, local_frac=0.65),

    6: dict(kappa=1.3, n_cand=12000, min_dist=0.05, local=0.14, local_frac=0.55),

    7: dict(kappa=1.3, n_cand=12000, min_dist=0.06, local=0.14, local_frac=0.50),

    8: dict(kappa=1.2, n_cand=15000, min_dist=0.08, local=0.12, local_frac=0.55),

}



def format_query(x):

    x = np.clip(np.asarray(x, float).ravel(), 1e-6, 1.0 - 1e-6)

    return "-".join(f"{v:.6f}" for v in x)



def lhs(n, d, rng):

    u = np.empty((n, d))

    edges = np.linspace(0.0, 1.0, n + 1)

    low, high = edges[:-1], edges[1:]

    for j in range(d):

        u[:, j] = rng.uniform(low, high)

        rng.shuffle(u[:, j])

    return u



queries = {}

for k in range(1, 9):

    X0 = np.load(base / f"function_{k}" / "initial_inputs.npy").astype(float)

    y0 = np.load(base / f"function_{k}" / "initial_outputs.npy").astype(float).ravel()

    X = np.vstack([X0, WEEK1_X[k]])

    y = np.append(y0, WEEK1_Y[k])

    d, pol, rng = DIMS[k], POLICY[k], np.random.default_rng(2000 + k)

    print(f"Fitting F{k}  n={len(y)}  best={y.max():.6g}  last={y[-1]:.6g}")



    xsc = StandardScaler().fit(X)

    kernel = (ConstantKernel(1.0, (1e-3, 1e3))

              * Matern(length_scale=np.full(d, 0.3), length_scale_bounds=(0.05, 8.0), nu=2.5)

              + WhiteKernel(1e-5, (1e-12, 1e-1)))

    gp = GaussianProcessRegressor(kernel=kernel, normalize_y=True,

                                  n_restarts_optimizer=2, random_state=k, alpha=1e-8)

    gp.fit(xsc.transform(X), y)



    n_local = int(pol["n_cand"] * pol["local_frac"])

    n_global = pol["n_cand"] - n_local

    cand = lhs(max(n_global, 1), d, rng)

    if n_local:

        noise = rng.normal(0.0, pol["local"], size=(n_local, d))

        cand = np.vstack([cand, np.clip(WEEK1_X[k] + noise, 1e-6, 1 - 1e-6)])



    mu, std = gp.predict(xsc.transform(cand), return_std=True)

    scores = mu + pol["kappa"] * std

    dist = np.min(np.linalg.norm(cand[:, None, :] - X[None, :, :], axis=2), axis=1)

    scores = np.where(dist < pol["min_dist"], -np.inf, scores)

    q = format_query(cand[int(np.argmax(scores))])

    queries[k] = q

    print(f"  SUBMIT: {q}\n")



print("=" * 60)

print("WEEK 2 — PASTE INTO THE PORTAL")

print("=" * 60)

for k, q in queries.items():

    print(f"Function {k}: {q}")



Fitting F1  n=11  best=7.71088e-16  last=-1.18148e-270
  SUBMIT: 0.520875-0.334357

Fitting F2  n=11  best=0.652585  last=0.652585
  SUBMIT: 0.701530-0.047056



C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified upper bound 8.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 8.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


Fitting F3  n=16  best=-0.0192728  last=-0.0192728
  SUBMIT: 0.879583-0.006977-0.001530

Fitting F4  n=31  best=-0.917425  last=-0.917425


C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified upper bound 8.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 8.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


  SUBMIT: 0.421199-0.400087-0.337912-0.436790

Fitting F5  n=21  best=2830.6  last=2830.6
  SUBMIT: 0.116598-0.999999-0.999999-0.999999

Fitting F6  n=21  best=-0.419956  last=-0.419956


C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified upper bound 8.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 8.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


  SUBMIT: 0.425618-0.196929-0.846445-0.972258-0.141083

Fitting F7  n=31  best=1.51456  last=1.51456


C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 8.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 2 of parameter k1__k2__length_scale is close to the specified upper bound 8.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


  SUBMIT: 0.000001-0.312636-0.151505-0.026906-0.361543-0.731254

Fitting F8  n=41  best=9.74437  last=9.74437
  SUBMIT: 0.102526-0.102909-0.008115-0.245476-0.999999-0.523852-0.248637-0.499859

WEEK 2 — PASTE INTO THE PORTAL
Function 1: 0.520875-0.334357
Function 2: 0.701530-0.047056
Function 3: 0.879583-0.006977-0.001530
Function 4: 0.421199-0.400087-0.337912-0.436790
Function 5: 0.116598-0.999999-0.999999-0.999999
Function 6: 0.425618-0.196929-0.846445-0.972258-0.141083
Function 7: 0.000001-0.312636-0.151505-0.026906-0.361543-0.731254
Function 8: 0.102526-0.102909-0.008115-0.245476-0.999999-0.523852-0.248637-0.499859


C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 3 of parameter k1__k2__length_scale is close to the specified upper bound 8.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 4 of parameter k1__k2__length_scale is close to the specified upper bound 8.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 5 of parameter k1__k2__length_scale is close to the specified upper bound 8.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
C:\Users\R

In [1]:
import numpy as np

from pathlib import Path

from sklearn.gaussian_process import GaussianProcessRegressor

from sklearn.gaussian_process.kernels import ConstantKernel, Matern, WhiteKernel

from sklearn.preprocessing import StandardScaler

from sklearn.svm import SVC



base = Path(r"C:\Users\Ramnath Lakshmanan\Downloads\Initial_data_points_starter (2)\initial_data")

DIMS = {1: 2, 2: 2, 3: 3, 4: 4, 5: 4, 6: 5, 7: 6, 8: 8}



W1X = {

    1: np.array([0.973822, 0.110674]),

    2: np.array([0.711216, 0.383626]),

    3: np.array([0.628013, 0.359224, 0.539509]),

    4: np.array([0.367191, 0.464182, 0.391040, 0.448820]),

    5: np.array([0.245752, 0.932264, 0.962162, 0.965058]),

    6: np.array([0.446820, 0.321047, 0.436584, 0.762321, 0.003307]),

    7: np.array([0.086332, 0.371883, 0.229157, 0.106524, 0.394874, 0.689739]),

    8: np.array([0.054501, 0.081016, 0.320114, 0.398440, 0.621808, 0.649205, 0.338643, 0.601487]),

}

W1Y = {1: -1.1814831817160104e-270, 2: 0.6525848308450871, 3: -0.01927280144331694,

       4: -0.9174251874710708, 5: 2830.5996301272953, 6: -0.41995556157505487,

       7: 1.5145555220314069, 8: 9.7443661466821}

W2X = {

    1: np.array([0.520875, 0.334357]),

    2: np.array([0.701530, 0.047056]),

    3: np.array([0.879583, 0.006977, 0.001530]),

    4: np.array([0.421199, 0.400087, 0.337912, 0.436790]),

    5: np.array([0.116598, 0.999999, 0.999999, 0.999999]),

    6: np.array([0.425618, 0.196929, 0.846445, 0.972258, 0.141083]),

    7: np.array([0.000001, 0.312636, 0.151505, 0.026906, 0.361543, 0.731254]),

    8: np.array([0.102526, 0.102909, 0.008115, 0.245476, 0.999999, 0.523852, 0.248637, 0.499859]),

}

W2Y = {1: 4.584141793749554e-13, 2: 0.6685300747584026, 3: -0.18176416202017948,

       4: 0.2852616086635966, 5: 4441.586450664394, 6: -0.5975592068881134,

       7: 1.0217094038661905, 8: 9.9177834946854}



# exploit winners; pull losers back toward their personal best

ANCHOR = {1: W2X[1], 2: W2X[2], 3: W1X[3], 4: W2X[4],

          5: W2X[5], 6: W1X[6], 7: W1X[7], 8: W2X[8]}

POLICY = {

    1: dict(kappa=2.0, local=0.12, local_frac=0.50, min_dist=0.03),

    2: dict(kappa=1.0, local=0.08, local_frac=0.70, min_dist=0.02),

    3: dict(kappa=1.2, local=0.10, local_frac=0.65, min_dist=0.03),

    4: dict(kappa=1.0, local=0.08, local_frac=0.70, min_dist=0.03),

    5: dict(kappa=0.8, local=0.06, local_frac=0.75, min_dist=0.02),

    6: dict(kappa=1.2, local=0.10, local_frac=0.65, min_dist=0.04),

    7: dict(kappa=1.2, local=0.10, local_frac=0.60, min_dist=0.05),

    8: dict(kappa=1.0, local=0.08, local_frac=0.65, min_dist=0.06),

}



def fmt(x):

    x = np.clip(np.asarray(x, float).ravel(), 1e-6, 1 - 1e-6)

    return "-".join(f"{v:.6f}" for v in x)



queries = {}

for k in range(1, 9):

    X = np.vstack([np.load(base / f"function_{k}" / "initial_inputs.npy").astype(float), W1X[k], W2X[k]])

    y = np.concatenate([np.load(base / f"function_{k}" / "initial_outputs.npy").astype(float).ravel(),

                        [W1Y[k], W2Y[k]]])

    d, pol, rng = DIMS[k], POLICY[k], np.random.default_rng(3000 + k)

    print(f"F{k} n={len(y)} best={y.max():.6g} last={y[-1]:.6g}")



    xsc = StandardScaler().fit(X)

    Xs = xsc.transform(X)

    gp = GaussianProcessRegressor(

        kernel=(ConstantKernel(1.0, (1e-3, 1e3))

                * Matern(np.full(d, 0.3), (0.05, 8.0), nu=2.5)

                + WhiteKernel(1e-5, (1e-12, 1e-1))),

        normalize_y=True, n_restarts_optimizer=2, random_state=k,

    )

    gp.fit(Xs, y)



    # soft-margin RBF SVM: high = y >= median

    lab = (y >= np.median(y)).astype(int)

    svm = SVC(kernel="rbf", C=1.0, gamma="scale", probability=True)

    try:

        svm.fit(Xs, lab)

        svm_ok = len(np.unique(lab)) == 2

    except Exception:

        svm_ok = False



    n_loc = int(12000 * pol["local_frac"])

    cand = np.clip(ANCHOR[k] + rng.normal(0, pol["local"], size=(n_loc, d)), 1e-6, 1 - 1e-6)

    cand = np.vstack([cand, rng.uniform(1e-6, 1 - 1e-6, size=(12000 - n_loc, d))])



    mu, std = gp.predict(xsc.transform(cand), return_std=True)

    scores = mu + pol["kappa"] * std

    if svm_ok:

        p_high = svm.predict_proba(xsc.transform(cand))[:, 1]

        scores = scores + 0.25 * (p_high - 0.5)   # small SVM bonus

    dist = np.min(np.linalg.norm(cand[:, None, :] - X[None, :, :], axis=2), axis=1)

    scores = np.where(dist < pol["min_dist"], -np.inf, scores)

    q = fmt(cand[int(np.argmax(scores))])

    queries[k] = q

    print("  SUBMIT:", q, "\n")



print("=" * 60)

print("WEEK 3 — PASTE INTO THE PORTAL")

for k, q in queries.items():

    print(f"Function {k}: {q}")

F1 n=12 best=4.58414e-13 last=4.58414e-13
  SUBMIT: 0.605824-0.805874 

F2 n=12 best=0.66853 last=0.66853


C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified upper bound 8.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 8.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
C:\Users\Ramnath Lakshmanan\anaconda3\New

  SUBMIT: 0.719374-0.085613 

F3 n=17 best=-0.0192728 last=-0.181764
  SUBMIT: 0.040729-0.915411-0.504944 

F4 n=32 best=0.285262 last=0.285262


C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified upper bound 8.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 3 of parameter k1__k2__length_scale is close to the specified upper bound 8.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
C:\Users\Ramnath Lakshmanan\anaconda3\New

  SUBMIT: 0.439246-0.393165-0.273345-0.439606 

F5 n=22 best=4441.59 last=4441.59
  SUBMIT: 0.000001-0.999999-0.999999-0.999999 

F6 n=22 best=-0.419956 last=-0.597559


C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 8.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 2 of parameter k1__k2__length_scale is close to the specified upper bound 8.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


  SUBMIT: 0.508487-0.271802-0.367910-0.999999-0.000001 

F7 n=32 best=1.51456 last=1.02171


C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


  SUBMIT: 0.078771-0.282330-0.246212-0.164149-0.401057-0.641158 

F8 n=42 best=9.91778 last=9.91778
  SUBMIT: 0.179705-0.000001-0.204942-0.149746-0.990645-0.489105-0.181870-0.440792 

WEEK 3 — PASTE INTO THE PORTAL
Function 1: 0.605824-0.805874
Function 2: 0.719374-0.085613
Function 3: 0.040729-0.915411-0.504944
Function 4: 0.439246-0.393165-0.273345-0.439606
Function 5: 0.000001-0.999999-0.999999-0.999999
Function 6: 0.508487-0.271802-0.367910-0.999999-0.000001
Function 7: 0.078771-0.282330-0.246212-0.164149-0.401057-0.641158
Function 8: 0.179705-0.000001-0.204942-0.149746-0.990645-0.489105-0.181870-0.440792


C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 3 of parameter k1__k2__length_scale is close to the specified upper bound 8.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 4 of parameter k1__k2__length_scale is close to the specified upper bound 8.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 5 of parameter k1__k2__length_scale is close to the specified upper bound 8.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
C:\Users\R

In [1]:
import numpy as np

from pathlib import Path

from sklearn.gaussian_process import GaussianProcessRegressor

from sklearn.gaussian_process.kernels import ConstantKernel, Matern, WhiteKernel

from sklearn.preprocessing import StandardScaler

from sklearn.neural_network import MLPRegressor



base = Path(r"C:\Users\Ramnath Lakshmanan\Downloads\Initial_data_points_starter (2)\initial_data")

DIMS = {1: 2, 2: 2, 3: 3, 4: 4, 5: 4, 6: 5, 7: 6, 8: 8}



W1X = {

    1: np.array([0.973822, 0.110674]),

    2: np.array([0.711216, 0.383626]),

    3: np.array([0.628013, 0.359224, 0.539509]),

    4: np.array([0.367191, 0.464182, 0.391040, 0.448820]),

    5: np.array([0.245752, 0.932264, 0.962162, 0.965058]),

    6: np.array([0.446820, 0.321047, 0.436584, 0.762321, 0.003307]),

    7: np.array([0.086332, 0.371883, 0.229157, 0.106524, 0.394874, 0.689739]),

    8: np.array([0.054501, 0.081016, 0.320114, 0.398440, 0.621808, 0.649205, 0.338643, 0.601487]),

}

W1Y = {1:-1.1814831817160104e-270,2:0.6525848308450871,3:-0.01927280144331694,

       4:-0.9174251874710708,5:2830.5996301272953,6:-0.41995556157505487,

       7:1.5145555220314069,8:9.7443661466821}

W2X = {

    1: np.array([0.520875, 0.334357]),

    2: np.array([0.701530, 0.047056]),

    3: np.array([0.879583, 0.006977, 0.001530]),

    4: np.array([0.421199, 0.400087, 0.337912, 0.436790]),

    5: np.array([0.116598, 0.999999, 0.999999, 0.999999]),

    6: np.array([0.425618, 0.196929, 0.846445, 0.972258, 0.141083]),

    7: np.array([0.000001, 0.312636, 0.151505, 0.026906, 0.361543, 0.731254]),

    8: np.array([0.102526, 0.102909, 0.008115, 0.245476, 0.999999, 0.523852, 0.248637, 0.499859]),

}

W2Y = {1:4.584141793749554e-13,2:0.6685300747584026,3:-0.18176416202017948,

       4:0.2852616086635966,5:4441.586450664394,6:-0.5975592068881134,

       7:1.0217094038661905,8:9.9177834946854}

W3X = {

    1: np.array([0.605824, 0.805874]),

    2: np.array([0.719374, 0.085613]),

    3: np.array([0.040729, 0.915411, 0.504944]),

    4: np.array([0.439246, 0.393165, 0.273345, 0.439606]),

    5: np.array([0.000001, 0.999999, 0.999999, 0.999999]),

    6: np.array([0.508487, 0.271802, 0.367910, 0.999999, 0.000001]),

    7: np.array([0.078771, 0.282330, 0.246212, 0.164149, 0.401057, 0.641158]),

    8: np.array([0.179705, 0.000001, 0.204942, 0.149746, 0.990645, 0.489105, 0.181870, 0.440792]),

}

W3Y = {1:5.221216935504337e-23,2:0.6685229247231236,3:-0.030942339134816123,

       4:-1.2975118969582797,5:4440.480873839292,6:-0.7811647047830718,

       7:1.9868235528281357,8:9.92646197977771}



# snap to the point that currently holds the best y

ANCHOR = {1: W2X[1], 2: W2X[2], 3: W1X[3], 4: W2X[4],

          5: W2X[5], 6: W1X[6], 7: W3X[7], 8: W3X[8]}

POLICY = {

    1: dict(kappa=1.8, local=0.08, min_dist=0.025, step=0.04),

    2: dict(kappa=0.9, local=0.05, min_dist=0.015, step=0.03),

    3: dict(kappa=1.1, local=0.08, min_dist=0.03, step=0.04),

    4: dict(kappa=1.0, local=0.06, min_dist=0.025, step=0.03),

    5: dict(kappa=0.7, local=0.05, min_dist=0.02, step=0.03),

    6: dict(kappa=1.1, local=0.08, min_dist=0.04, step=0.04),

    7: dict(kappa=1.0, local=0.07, min_dist=0.04, step=0.03),

    8: dict(kappa=0.9, local=0.06, min_dist=0.05, step=0.03),

}



def fmt(x):

    x = np.clip(np.asarray(x, float).ravel(), 1e-6, 1-1e-6)

    return "-".join(f"{v:.6f}" for v in x)



def fd_grad(model, scaler, x, eps=1e-3):

    g = np.zeros(x.size)

    x = x.reshape(1, -1)

    for j in range(x.size):

        xp, xm = x.copy(), x.copy()

        xp[0, j] += eps; xm[0, j] -= eps

        g[j] = (model.predict(scaler.transform(np.clip(xp, 0, 1))) -

                model.predict(scaler.transform(np.clip(xm, 0, 1))))[0] / (2*eps)

    return g



queries = {}

for k in range(1, 9):

    X = np.vstack([np.load(base/f"function_{k}"/"initial_inputs.npy").astype(float),

                   W1X[k], W2X[k], W3X[k]])

    y = np.concatenate([np.load(base/f"function_{k}"/"initial_outputs.npy").astype(float).ravel(),

                        [W1Y[k], W2Y[k], W3Y[k]]])

    d, pol, rng = DIMS[k], POLICY[k], np.random.default_rng(4000+k)

    print(f"F{k} n={len(y)} best={y.max():.6g} last={y[-1]:.6g}")



    xsc = StandardScaler().fit(X)

    gp = GaussianProcessRegressor(

        kernel=(ConstantKernel(1.0,(1e-3,1e3))*Matern(np.full(d,0.3),(0.05,8.0),nu=2.5)

                +WhiteKernel(1e-5,(1e-12,1e-1))),

        normalize_y=True, n_restarts_optimizer=2, random_state=k)

    gp.fit(xsc.transform(X), y)



    # tiny MLP for a gradient hint only

    ysc = StandardScaler().fit(y.reshape(-1,1))

    mlp = MLPRegressor(hidden_layer_sizes=(16,8), activation="relu",

                       max_iter=2000, random_state=k, alpha=0.01)

    try:

        mlp.fit(xsc.transform(X), ysc.transform(y.reshape(-1,1)).ravel())

        g = fd_grad(mlp, xsc, ANCHOR[k])

        print("  MLP grad (raw):", np.round(g, 3))

        step = ANCHOR[k] + pol["step"] * (g / (np.linalg.norm(g)+1e-9))

        step = np.clip(step, 1e-6, 1-1e-6)

    except Exception as e:

        print("  MLP skipped:", e)

        step = ANCHOR[k]



    n_loc = 8000

    cand = np.clip(ANCHOR[k] + rng.normal(0, pol["local"], size=(n_loc, d)), 1e-6, 1-1e-6)

    cand = np.vstack([cand, step.reshape(1,-1),

                      rng.uniform(1e-6, 1-1e-6, size=(4000, d))])

    mu, std = gp.predict(xsc.transform(cand), return_std=True)

    scores = mu + pol["kappa"]*std

    dist = np.min(np.linalg.norm(cand[:,None,:]-X[None,:,:], axis=2), axis=1)

    scores = np.where(dist < pol["min_dist"], -np.inf, scores)

    q = fmt(cand[int(np.argmax(scores))])

    queries[k] = q

    print("  SUBMIT:", q, "\n")



print("="*60)

print("WEEK 4 — PASTE INTO THE PORTAL")

for k,q in queries.items():

    print(f"Function {k}: {q}")

F1 n=13 best=4.58414e-13 last=5.22122e-23


C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified upper bound 8.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


  MLP grad (raw): [-2.834 -3.183]
  SUBMIT: 0.906435-0.524773 

F2 n=13 best=0.66853 last=0.668523


C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 8.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


  MLP grad (raw): [6.141 0.766]
  SUBMIT: 0.711541-0.000001 

F3 n=18 best=-0.0192728 last=-0.0309423


...

  MLP grad (raw): [-5.495  0.948  1.878  0.438]
  SUBMIT: 0.070905-0.999999-0.999999-0.812090 

F6 n=23 best=-0.419956 last=-0.781165
  MLP grad (raw): [-0.403  1.149  1.249  0.441 -1.318]
  SUBMIT: 0.423685-0.314205-0.861819-0.607621-0.009157 

F7 n=33 best=1.98682 last=1.98682


C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 8.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 2 of parameter k1__k2__length_scale is close to the specified upper bound 8.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


  MLP grad (raw): [-1.966 -1.271 -0.438 -1.378 -0.663  1.045]
  SUBMIT: 0.053889-0.170531-0.230571-0.223537-0.393017-0.615148 

F8 n=43 best=9.92646 last=9.92646


C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 3 of parameter k1__k2__length_scale is close to the specified upper bound 8.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 4 of parameter k1__k2__length_scale is close to the specified upper bound 8.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 5 of parameter k1__k2__length_scale is close to the specified upper bound 8.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
C:\Users\R

  MLP grad (raw): [-0.55  -0.628 -0.126 -0.173  0.906 -0.21  -0.425 -0.037]
  SUBMIT: 0.000001-0.000001-0.058234-0.160975-0.905106-0.422655-0.072201-0.456271 

WEEK 4 — PASTE INTO THE PORTAL
Function 1: 0.906435-0.524773
Function 2: 0.711541-0.000001
Function 3: 0.901391-0.100657-0.501946
Function 4: 0.404491-0.411487-0.374128-0.419096
Function 5: 0.070905-0.999999-0.999999-0.812090
Function 6: 0.423685-0.314205-0.861819-0.607621-0.009157
Function 7: 0.053889-0.170531-0.230571-0.223537-0.393017-0.615148
Function 8: 0.000001-0.000001-0.058234-0.160975-0.905106-0.422655-0.072201-0.456271


In [1]:
import numpy as np

from pathlib import Path

from sklearn.gaussian_process import GaussianProcessRegressor

from sklearn.gaussian_process.kernels import ConstantKernel, Matern, WhiteKernel

from sklearn.preprocessing import StandardScaler

from sklearn.neural_network import MLPRegressor



base = Path(r"C:\Users\Ramnath Lakshmanan\Downloads\Initial_data_points_starter (2)\initial_data")

DIMS = {1: 2, 2: 2, 3: 3, 4: 4, 5: 4, 6: 5, 7: 6, 8: 8}



W1X = {

    1: np.array([0.973822, 0.110674]),

    2: np.array([0.711216, 0.383626]),

    3: np.array([0.628013, 0.359224, 0.539509]),

    4: np.array([0.367191, 0.464182, 0.391040, 0.448820]),

    5: np.array([0.245752, 0.932264, 0.962162, 0.965058]),

    6: np.array([0.446820, 0.321047, 0.436584, 0.762321, 0.003307]),

    7: np.array([0.086332, 0.371883, 0.229157, 0.106524, 0.394874, 0.689739]),

    8: np.array([0.054501, 0.081016, 0.320114, 0.398440, 0.621808, 0.649205, 0.338643, 0.601487]),

}

W1Y = {1:-1.1814831817160104e-270,2:0.6525848308450871,3:-0.01927280144331694,

       4:-0.9174251874710708,5:2830.5996301272953,6:-0.41995556157505487,

       7:1.5145555220314069,8:9.7443661466821}

W2X = {

    1: np.array([0.520875, 0.334357]),

    2: np.array([0.701530, 0.047056]),

    3: np.array([0.879583, 0.006977, 0.001530]),

    4: np.array([0.421199, 0.400087, 0.337912, 0.436790]),

    5: np.array([0.116598, 0.999999, 0.999999, 0.999999]),

    6: np.array([0.425618, 0.196929, 0.846445, 0.972258, 0.141083]),

    7: np.array([0.000001, 0.312636, 0.151505, 0.026906, 0.361543, 0.731254]),

    8: np.array([0.102526, 0.102909, 0.008115, 0.245476, 0.999999, 0.523852, 0.248637, 0.499859]),

}

W2Y = {1:4.584141793749554e-13,2:0.6685300747584026,3:-0.18176416202017948,

       4:0.2852616086635966,5:4441.586450664394,6:-0.5975592068881134,

       7:1.0217094038661905,8:9.9177834946854}

W3X = {

    1: np.array([0.605824, 0.805874]),

    2: np.array([0.719374, 0.085613]),

    3: np.array([0.040729, 0.915411, 0.504944]),

    4: np.array([0.439246, 0.393165, 0.273345, 0.439606]),

    5: np.array([0.000001, 0.999999, 0.999999, 0.999999]),

    6: np.array([0.508487, 0.271802, 0.367910, 0.999999, 0.000001]),

    7: np.array([0.078771, 0.282330, 0.246212, 0.164149, 0.401057, 0.641158]),

    8: np.array([0.179705, 0.000001, 0.204942, 0.149746, 0.990645, 0.489105, 0.181870, 0.440792]),

}

W3Y = {1:5.221216935504337e-23,2:0.6685229247231236,3:-0.030942339134816123,

       4:-1.2975118969582797,5:4440.480873839292,6:-0.7811647047830718,

       7:1.9868235528281357,8:9.92646197977771}

W4X = {

    1: np.array([0.906435, 0.524773]),

    2: np.array([0.711541, 0.000001]),

    3: np.array([0.901391, 0.100657, 0.501946]),

    4: np.array([0.404491, 0.411487, 0.374128, 0.419096]),

    5: np.array([0.070905, 0.999999, 0.999999, 0.812090]),

    6: np.array([0.423685, 0.314205, 0.861819, 0.607621, 0.009157]),

    7: np.array([0.053889, 0.170531, 0.230571, 0.223537, 0.393017, 0.615148]),

    8: np.array([0.000001, 0.000001, 0.058234, 0.160975, 0.905106, 0.422655, 0.072201, 0.456271]),

}

W4Y = {1:-9.171561473040102e-63,2:0.718243930604028,3:-0.05350002457722131,

       4:0.6098506102572618,5:2690.3948413647527,6:-0.6049887641373398,

       7:2.2735115338126204,8:9.8956923171149}



ANCHOR = {1: W2X[1], 2: W4X[2], 3: W1X[3], 4: W4X[4],

          5: W2X[5], 6: W1X[6], 7: W4X[7], 8: W3X[8]}

POLICY = {

    1: dict(kappa=1.8, local=0.08, min_dist=0.025),

    2: dict(kappa=0.8, local=0.04, min_dist=0.012),

    3: dict(kappa=1.1, local=0.08, min_dist=0.03),

    4: dict(kappa=0.8, local=0.05, min_dist=0.02),

    5: dict(kappa=0.7, local=0.04, min_dist=0.02),

    6: dict(kappa=1.1, local=0.08, min_dist=0.04),

    7: dict(kappa=0.8, local=0.06, min_dist=0.03),

    8: dict(kappa=0.9, local=0.06, min_dist=0.05),

}



def fmt(x):

    x = np.clip(np.asarray(x, float).ravel(), 1e-6, 1 - 1e-6)

    return "-".join(f"{v:.6f}" for v in x)



queries = {}

for k in range(1, 9):

    X0 = np.load(base / f"function_{k}" / "initial_inputs.npy").astype(float)

    y0 = np.load(base / f"function_{k}" / "initial_outputs.npy").astype(float).ravel()

    X = np.vstack([X0, W1X[k], W2X[k], W3X[k], W4X[k]])

    y = np.concatenate([y0, [W1Y[k], W2Y[k], W3Y[k], W4Y[k]]])

    d, pol, rng = DIMS[k], POLICY[k], np.random.default_rng(5000 + k)

    print(f"F{k} n={len(y)} best={y.max():.6g} last={y[-1]:.6g}")



    xsc = StandardScaler().fit(X)

    gp = GaussianProcessRegressor(

        kernel=(ConstantKernel(1.0, (1e-3, 1e3))

                * Matern(np.full(d, 0.3), (0.05, 8.0), nu=2.5)

                + WhiteKernel(1e-5, (1e-12, 1e-1))),

        normalize_y=True, n_restarts_optimizer=2, random_state=k,

    )

    gp.fit(xsc.transform(X), y)



    cand = np.clip(ANCHOR[k] + rng.normal(0, pol["local"], size=(9000, d)), 1e-6, 1-1e-6)

    cand = np.vstack([cand, rng.uniform(1e-6, 1-1e-6, size=(3000, d))])

    mu, std = gp.predict(xsc.transform(cand), return_std=True)

    scores = mu + pol["kappa"] * std

    dist = np.min(np.linalg.norm(cand[:, None, :] - X[None, :, :], axis=2), axis=1)

    scores = np.where(dist < pol["min_dist"], -np.inf, scores)

    q = fmt(cand[int(np.argmax(scores))])

    queries[k] = q

    print("  SUBMIT:", q, "\n")



print("=" * 60)

print("WEEK 5 — PASTE INTO THE PORTAL")

print("=" * 60)

for k, q in queries.items():

    print(f"Function {k}: {q}")

F1 n=14 best=4.58414e-13 last=-9.17156e-63
  SUBMIT: 0.276310-0.463966 

F2 n=14 best=0.718244 last=0.718244


C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified upper bound 8.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 8.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


  SUBMIT: 0.711701-0.012157 

F3 n=19 best=-0.0192728 last=-0.0535
  SUBMIT: 0.812590-0.971252-0.407756 



C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified upper bound 8.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 8.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


F4 n=34 best=0.609851 last=0.609851
  SUBMIT: 0.449239-0.320819-0.430682-0.426303 

F5 n=24 best=4441.59 last=2690.39


C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified upper bound 8.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


  SUBMIT: 0.029536-0.999999-0.999999-0.999999 

F6 n=24 best=-0.419956 last=-0.604989
  SUBMIT: 0.440177-0.307928-0.516852-0.718082-0.163715 

F7 n=34 best=2.27351 last=2.27351


C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 8.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 2 of parameter k1__k2__length_scale is close to the specified upper bound 8.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


  SUBMIT: 0.000001-0.079068-0.203789-0.237496-0.371176-0.578991 

F8 n=44 best=9.92646 last=9.89569
  SUBMIT: 0.116622-0.061321-0.145257-0.029806-0.891611-0.487268-0.132540-0.643656 

WEEK 5 — PASTE INTO THE PORTAL
Function 1: 0.276310-0.463966
Function 2: 0.711701-0.012157
Function 3: 0.812590-0.971252-0.407756
Function 4: 0.449239-0.320819-0.430682-0.426303
Function 5: 0.029536-0.999999-0.999999-0.999999
Function 6: 0.440177-0.307928-0.516852-0.718082-0.163715
Function 7: 0.000001-0.079068-0.203789-0.237496-0.371176-0.578991
Function 8: 0.116622-0.061321-0.145257-0.029806-0.891611-0.487268-0.132540-0.643656


C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 3 of parameter k1__k2__length_scale is close to the specified upper bound 8.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 4 of parameter k1__k2__length_scale is close to the specified upper bound 8.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 5 of parameter k1__k2__length_scale is close to the specified upper bound 8.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
C:\Users\R

In [1]:
import numpy as np

from pathlib import Path

from sklearn.gaussian_process import GaussianProcessRegressor

from sklearn.gaussian_process.kernels import ConstantKernel, Matern, WhiteKernel

from sklearn.preprocessing import StandardScaler



base = Path(r"C:\Users\Ramnath Lakshmanan\Downloads\Initial_data_points_starter (2)\initial_data")

DIMS = {1: 2, 2: 2, 3: 3, 4: 4, 5: 4, 6: 5, 7: 6, 8: 8}



W1X = {

    1: np.array([0.973822, 0.110674]),

    2: np.array([0.711216, 0.383626]),

    3: np.array([0.628013, 0.359224, 0.539509]),

    4: np.array([0.367191, 0.464182, 0.391040, 0.448820]),

    5: np.array([0.245752, 0.932264, 0.962162, 0.965058]),

    6: np.array([0.446820, 0.321047, 0.436584, 0.762321, 0.003307]),

    7: np.array([0.086332, 0.371883, 0.229157, 0.106524, 0.394874, 0.689739]),

    8: np.array([0.054501, 0.081016, 0.320114, 0.398440, 0.621808, 0.649205, 0.338643, 0.601487]),

}

W1Y = {1:-1.1814831817160104e-270,2:0.6525848308450871,3:-0.01927280144331694,

       4:-0.9174251874710708,5:2830.5996301272953,6:-0.41995556157505487,

       7:1.5145555220314069,8:9.7443661466821}

W2X = {

    1: np.array([0.520875, 0.334357]),

    2: np.array([0.701530, 0.047056]),

    3: np.array([0.879583, 0.006977, 0.001530]),

    4: np.array([0.421199, 0.400087, 0.337912, 0.436790]),

    5: np.array([0.116598, 0.999999, 0.999999, 0.999999]),

    6: np.array([0.425618, 0.196929, 0.846445, 0.972258, 0.141083]),

    7: np.array([0.000001, 0.312636, 0.151505, 0.026906, 0.361543, 0.731254]),

    8: np.array([0.102526, 0.102909, 0.008115, 0.245476, 0.999999, 0.523852, 0.248637, 0.499859]),

}

W2Y = {1:4.584141793749554e-13,2:0.6685300747584026,3:-0.18176416202017948,

       4:0.2852616086635966,5:4441.586450664394,6:-0.5975592068881134,

       7:1.0217094038661905,8:9.9177834946854}

W3X = {

    1: np.array([0.605824, 0.805874]),

    2: np.array([0.719374, 0.085613]),

    3: np.array([0.040729, 0.915411, 0.504944]),

    4: np.array([0.439246, 0.393165, 0.273345, 0.439606]),

    5: np.array([0.000001, 0.999999, 0.999999, 0.999999]),

    6: np.array([0.508487, 0.271802, 0.367910, 0.999999, 0.000001]),

    7: np.array([0.078771, 0.282330, 0.246212, 0.164149, 0.401057, 0.641158]),

    8: np.array([0.179705, 0.000001, 0.204942, 0.149746, 0.990645, 0.489105, 0.181870, 0.440792]),

}

W3Y = {1:5.221216935504337e-23,2:0.6685229247231236,3:-0.030942339134816123,

       4:-1.2975118969582797,5:4440.480873839292,6:-0.7811647047830718,

       7:1.9868235528281357,8:9.92646197977771}

W4X = {

    1: np.array([0.906435, 0.524773]),

    2: np.array([0.711541, 0.000001]),

    3: np.array([0.901391, 0.100657, 0.501946]),

    4: np.array([0.404491, 0.411487, 0.374128, 0.419096]),

    5: np.array([0.070905, 0.999999, 0.999999, 0.812090]),

    6: np.array([0.423685, 0.314205, 0.861819, 0.607621, 0.009157]),

    7: np.array([0.053889, 0.170531, 0.230571, 0.223537, 0.393017, 0.615148]),

    8: np.array([0.000001, 0.000001, 0.058234, 0.160975, 0.905106, 0.422655, 0.072201, 0.456271]),

}

W4Y = {1:-9.171561473040102e-63,2:0.718243930604028,3:-0.05350002457722131,

       4:0.6098506102572618,5:2690.3948413647527,6:-0.6049887641373398,

       7:2.2735115338126204,8:9.8956923171149}

W5X = {

    1: np.array([0.276310, 0.463966]),

    2: np.array([0.711701, 0.012157]),

    3: np.array([0.812590, 0.971252, 0.407756]),

    4: np.array([0.449239, 0.320819, 0.430682, 0.426303]),

    5: np.array([0.029536, 0.999999, 0.999999, 0.999999]),

    6: np.array([0.440177, 0.307928, 0.516852, 0.718082, 0.163715]),

    7: np.array([0.000001, 0.079068, 0.203789, 0.237496, 0.371176, 0.578991]),

    8: np.array([0.116622, 0.061321, 0.145257, 0.029806, 0.891611, 0.487268, 0.132540, 0.643656]),

}

W5Y = {1:-1.4588117677798244e-17,2:0.7233854034475573,3:-0.0348256634794126,

       4:-0.3587979590239736,5:4440.508119101403,6:-0.3636516136843698,

       7:2.113146751542243,8:9.9627878480899}



ANCHOR = {1: W2X[1], 2: W5X[2], 3: W1X[3], 4: W4X[4],

          5: W2X[5], 6: W5X[6], 7: W4X[7], 8: W5X[8]}

POLICY = {

    1: dict(kappa=1.8, local=0.08, min_dist=0.025),

    2: dict(kappa=0.7, local=0.03, min_dist=0.01),

    3: dict(kappa=1.1, local=0.08, min_dist=0.03),

    4: dict(kappa=0.8, local=0.05, min_dist=0.02),

    5: dict(kappa=0.6, local=0.03, min_dist=0.015),

    6: dict(kappa=0.9, local=0.06, min_dist=0.03),

    7: dict(kappa=0.8, local=0.05, min_dist=0.03),

    8: dict(kappa=0.8, local=0.05, min_dist=0.04),

}



def fmt(x):

    x = np.clip(np.asarray(x, float).ravel(), 1e-6, 1-1e-6)

    return "-".join(f"{v:.6f}" for v in x)



queries = {}

for k in range(1, 9):

    X0 = np.load(base / f"function_{k}" / "initial_inputs.npy").astype(float)

    y0 = np.load(base / f"function_{k}" / "initial_outputs.npy").astype(float).ravel()

    X = np.vstack([X0, W1X[k], W2X[k], W3X[k], W4X[k], W5X[k]])

    y = np.concatenate([y0, [W1Y[k], W2Y[k], W3Y[k], W4Y[k], W5Y[k]]])

    d, pol, rng = DIMS[k], POLICY[k], np.random.default_rng(6000 + k)

    print(f"F{k} n={len(y)} best={y.max():.6g} last={y[-1]:.6g}")

    xsc = StandardScaler().fit(X)

    gp = GaussianProcessRegressor(

        kernel=(ConstantKernel(1.0, (1e-3, 1e3))

                * Matern(np.full(d, 0.3), (0.05, 8.0), nu=2.5)

                + WhiteKernel(1e-5, (1e-12, 1e-1))),

        normalize_y=True, n_restarts_optimizer=2, random_state=k)

    gp.fit(xsc.transform(X), y)

    cand = np.clip(ANCHOR[k] + rng.normal(0, pol["local"], size=(9000, d)), 1e-6, 1-1e-6)

    cand = np.vstack([cand, rng.uniform(1e-6, 1-1e-6, size=(3000, d))])

    mu, std = gp.predict(xsc.transform(cand), return_std=True)

    scores = mu + pol["kappa"] * std

    dist = np.min(np.linalg.norm(cand[:, None, :] - X[None, :, :], axis=2), axis=1)

    scores = np.where(dist < pol["min_dist"], -np.inf, scores)

    q = fmt(cand[int(np.argmax(scores))])

    queries[k] = q

    print("  SUBMIT:", q, "\n")



print("=" * 60)

print("WEEK 6 — PASTE INTO THE PORTAL")

print("=" * 60)

for k, q in queries.items():

    print(f"Function {k}: {q}")

F1 n=15 best=4.58414e-13 last=-1.45881e-17
  SUBMIT: 0.147012-0.207382 

F2 n=15 best=0.723385 last=0.723385
  SUBMIT: 0.711320-0.022518 

F3 n=20 best=-0.0192728 last=-0.0348257


C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified upper bound 8.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified upper bound 8.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 8.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


  SUBMIT: 0.042548-0.018175-0.535544 

F4 n=35 best=0.609851 last=-0.358798
  SUBMIT: 0.417683-0.392791-0.389865-0.415352 

F5 n=25 best=4441.59 last=4440.51


C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified upper bound 8.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


  SUBMIT: 0.072146-0.999999-0.999999-0.999999 

F6 n=25 best=-0.363652 last=-0.363652
  SUBMIT: 0.483442-0.406320-0.551572-0.728395-0.141301 

F7 n=35 best=2.27351 last=2.11315


C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 8.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 2 of parameter k1__k2__length_scale is close to the specified upper bound 8.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


  SUBMIT: 0.002952-0.137104-0.221381-0.265099-0.399907-0.616118 

F8 n=45 best=9.96279 last=9.96279
  SUBMIT: 0.024400-0.068670-0.123985-0.088530-0.958126-0.519233-0.251301-0.734513 

WEEK 6 — PASTE INTO THE PORTAL
Function 1: 0.147012-0.207382
Function 2: 0.711320-0.022518
Function 3: 0.042548-0.018175-0.535544
Function 4: 0.417683-0.392791-0.389865-0.415352
Function 5: 0.072146-0.999999-0.999999-0.999999
Function 6: 0.483442-0.406320-0.551572-0.728395-0.141301
Function 7: 0.002952-0.137104-0.221381-0.265099-0.399907-0.616118
Function 8: 0.024400-0.068670-0.123985-0.088530-0.958126-0.519233-0.251301-0.734513


C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 3 of parameter k1__k2__length_scale is close to the specified upper bound 8.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 4 of parameter k1__k2__length_scale is close to the specified upper bound 8.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 5 of parameter k1__k2__length_scale is close to the specified upper bound 8.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
C:\Users\R

In [1]:
import numpy as np

from pathlib import Path

from sklearn.gaussian_process import GaussianProcessRegressor

from sklearn.gaussian_process.kernels import ConstantKernel, Matern, WhiteKernel

from sklearn.preprocessing import StandardScaler



base = Path(r"C:\Users\Ramnath Lakshmanan\Downloads\Initial_data_points_starter (2)\initial_data")

DIMS = {1: 2, 2: 2, 3: 3, 4: 4, 5: 4, 6: 5, 7: 6, 8: 8}



W1X = {

    1: np.array([0.973822, 0.110674]), 2: np.array([0.711216, 0.383626]),

    3: np.array([0.628013, 0.359224, 0.539509]),

    4: np.array([0.367191, 0.464182, 0.391040, 0.448820]),

    5: np.array([0.245752, 0.932264, 0.962162, 0.965058]),

    6: np.array([0.446820, 0.321047, 0.436584, 0.762321, 0.003307]),

    7: np.array([0.086332, 0.371883, 0.229157, 0.106524, 0.394874, 0.689739]),

    8: np.array([0.054501, 0.081016, 0.320114, 0.398440, 0.621808, 0.649205, 0.338643, 0.601487]),

}

W1Y = {1:-1.1814831817160104e-270,2:0.6525848308450871,3:-0.01927280144331694,

       4:-0.9174251874710708,5:2830.5996301272953,6:-0.41995556157505487,

       7:1.5145555220314069,8:9.7443661466821}

W2X = {

    1: np.array([0.520875, 0.334357]), 2: np.array([0.701530, 0.047056]),

    3: np.array([0.879583, 0.006977, 0.001530]),

    4: np.array([0.421199, 0.400087, 0.337912, 0.436790]),

    5: np.array([0.116598, 0.999999, 0.999999, 0.999999]),

    6: np.array([0.425618, 0.196929, 0.846445, 0.972258, 0.141083]),

    7: np.array([0.000001, 0.312636, 0.151505, 0.026906, 0.361543, 0.731254]),

    8: np.array([0.102526, 0.102909, 0.008115, 0.245476, 0.999999, 0.523852, 0.248637, 0.499859]),

}

W2Y = {1:4.584141793749554e-13,2:0.6685300747584026,3:-0.18176416202017948,

       4:0.2852616086635966,5:4441.586450664394,6:-0.5975592068881134,

       7:1.0217094038661905,8:9.9177834946854}

W3X = {

    1: np.array([0.605824, 0.805874]), 2: np.array([0.719374, 0.085613]),

    3: np.array([0.040729, 0.915411, 0.504944]),

    4: np.array([0.439246, 0.393165, 0.273345, 0.439606]),

    5: np.array([0.000001, 0.999999, 0.999999, 0.999999]),

    6: np.array([0.508487, 0.271802, 0.367910, 0.999999, 0.000001]),

    7: np.array([0.078771, 0.282330, 0.246212, 0.164149, 0.401057, 0.641158]),

    8: np.array([0.179705, 0.000001, 0.204942, 0.149746, 0.990645, 0.489105, 0.181870, 0.440792]),

}

W3Y = {1:5.221216935504337e-23,2:0.6685229247231236,3:-0.030942339134816123,

       4:-1.2975118969582797,5:4440.480873839292,6:-0.7811647047830718,

       7:1.9868235528281357,8:9.92646197977771}

W4X = {

    1: np.array([0.906435, 0.524773]), 2: np.array([0.711541, 0.000001]),

    3: np.array([0.901391, 0.100657, 0.501946]),

    4: np.array([0.404491, 0.411487, 0.374128, 0.419096]),

    5: np.array([0.070905, 0.999999, 0.999999, 0.812090]),

    6: np.array([0.423685, 0.314205, 0.861819, 0.607621, 0.009157]),

    7: np.array([0.053889, 0.170531, 0.230571, 0.223537, 0.393017, 0.615148]),

    8: np.array([0.000001, 0.000001, 0.058234, 0.160975, 0.905106, 0.422655, 0.072201, 0.456271]),

}

W4Y = {1:-9.171561473040102e-63,2:0.718243930604028,3:-0.05350002457722131,

       4:0.6098506102572618,5:2690.3948413647527,6:-0.6049887641373398,

       7:2.2735115338126204,8:9.8956923171149}

W5X = {

    1: np.array([0.276310, 0.463966]), 2: np.array([0.711701, 0.012157]),

    3: np.array([0.812590, 0.971252, 0.407756]),

    4: np.array([0.449239, 0.320819, 0.430682, 0.426303]),

    5: np.array([0.029536, 0.999999, 0.999999, 0.999999]),

    6: np.array([0.440177, 0.307928, 0.516852, 0.718082, 0.163715]),

    7: np.array([0.000001, 0.079068, 0.203789, 0.237496, 0.371176, 0.578991]),

    8: np.array([0.116622, 0.061321, 0.145257, 0.029806, 0.891611, 0.487268, 0.132540, 0.643656]),

}

W5Y = {1:-1.4588117677798244e-17,2:0.7233854034475573,3:-0.0348256634794126,

       4:-0.3587979590239736,5:4440.508119101403,6:-0.3636516136843698,

       7:2.113146751542243,8:9.9627878480899}

W6X = {

    1: np.array([0.147012, 0.207382]), 2: np.array([0.711320, 0.022518]),

    3: np.array([0.042548, 0.018175, 0.535544]),

    4: np.array([0.417683, 0.392791, 0.389865, 0.415352]),

    5: np.array([0.072146, 0.999999, 0.999999, 0.999999]),

    6: np.array([0.483442, 0.406320, 0.551572, 0.728395, 0.141301]),

    7: np.array([0.002952, 0.137104, 0.221381, 0.265099, 0.399907, 0.616118]),

    8: np.array([0.024400, 0.068670, 0.123985, 0.088530, 0.958126, 0.519233, 0.251301, 0.734513]),

}

W6Y = {1:7.513182419081889e-87,2:0.5266505044433385,3:-0.10301859579872755,

       4:0.45948056013104877,5:4440.724001387989,6:-0.29074974600067394,

       7:2.1465849895947136,8:9.9581228253791}



ANCHOR = {1: W2X[1], 2: W5X[2], 3: W1X[3], 4: W4X[4],

          5: W2X[5], 6: W6X[6], 7: W4X[7], 8: W5X[8]}

POLICY = {

    1: dict(kappa=1.8, local=0.08, min_dist=0.025),

    2: dict(kappa=0.6, local=0.025, min_dist=0.008),

    3: dict(kappa=1.1, local=0.07, min_dist=0.03),

    4: dict(kappa=0.7, local=0.04, min_dist=0.018),

    5: dict(kappa=0.5, local=0.03, min_dist=0.015),

    6: dict(kappa=0.8, local=0.05, min_dist=0.025),

    7: dict(kappa=0.7, local=0.05, min_dist=0.025),

    8: dict(kappa=0.7, local=0.04, min_dist=0.035),

}



def fmt(x):

    x = np.clip(np.asarray(x, float).ravel(), 1e-6, 1-1e-6)

    return "-".join(f"{v:.6f}" for v in x)



queries = {}

for k in range(1, 9):

    X0 = np.load(base / f"function_{k}" / "initial_inputs.npy").astype(float)

    y0 = np.load(base / f"function_{k}" / "initial_outputs.npy").astype(float).ravel()

    X = np.vstack([X0, W1X[k], W2X[k], W3X[k], W4X[k], W5X[k], W6X[k]])

    y = np.concatenate([y0, [W1Y[k], W2Y[k], W3Y[k], W4Y[k], W5Y[k], W6Y[k]]])

    d, pol, rng = DIMS[k], POLICY[k], np.random.default_rng(7000 + k)

    print(f"F{k} n={len(y)} best={y.max():.6g} last={y[-1]:.6g}")

    xsc = StandardScaler().fit(X)

    gp = GaussianProcessRegressor(

        kernel=(ConstantKernel(1.0, (1e-3, 1e3))

                * Matern(np.full(d, 0.3), (0.05, 8.0), nu=2.5)

                + WhiteKernel(1e-5, (1e-12, 1e-1))),

        normalize_y=True, n_restarts_optimizer=2, random_state=k)

    gp.fit(xsc.transform(X), y)

    cand = np.clip(ANCHOR[k] + rng.normal(0, pol["local"], size=(9000, d)), 1e-6, 1-1e-6)

    cand = np.vstack([cand, rng.uniform(1e-6, 1-1e-6, size=(2500, d))])

    mu, std = gp.predict(xsc.transform(cand), return_std=True)

    scores = mu + pol["kappa"] * std

    dist = np.min(np.linalg.norm(cand[:, None, :] - X[None, :, :], axis=2), axis=1)

    scores = np.where(dist < pol["min_dist"], -np.inf, scores)

    q = fmt(cand[int(np.argmax(scores))])

    queries[k] = q

    print("  SUBMIT:", q, "\n")



print("=" * 60)

print("WEEK 7 — PASTE INTO THE PORTAL")

print("=" * 60)

for k, q in queries.items():

    print(f"Function {k}: {q}")

F1 n=16 best=4.58414e-13 last=7.51318e-87
  SUBMIT: 0.623326-0.011857 



C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified upper bound 8.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


F2 n=16 best=0.723385 last=0.526651
  SUBMIT: 0.688509-0.006758 

F3 n=21 best=-0.0192728 last=-0.103019
  SUBMIT: 0.973709-0.001813-0.792312 

F4 n=36 best=0.609851 last=0.459481
  SUBMIT: 0.411661-0.384885-0.399073-0.428315 

F5 n=26 best=4441.59 last=4440.72
  SUBMIT: 0.206499-0.999999-0.999999-0.999999 



C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified upper bound 8.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\gaussian_process\kernels.py:445: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-12. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


F6 n=26 best=-0.29075 last=-0.29075
  SUBMIT: 0.528613-0.408984-0.624132-0.735857-0.121553 

F7 n=36 best=2.27351 last=2.14658


C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 8.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 2 of parameter k1__k2__length_scale is close to the specified upper bound 8.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


  SUBMIT: 0.076466-0.107136-0.246213-0.221705-0.391060-0.600276 

F8 n=46 best=9.96279 last=9.95812
  SUBMIT: 0.123965-0.028657-0.123383-0.188126-0.900411-0.481502-0.091022-0.738327 

WEEK 7 — PASTE INTO THE PORTAL
Function 1: 0.623326-0.011857
Function 2: 0.688509-0.006758
Function 3: 0.973709-0.001813-0.792312
Function 4: 0.411661-0.384885-0.399073-0.428315
Function 5: 0.206499-0.999999-0.999999-0.999999
Function 6: 0.528613-0.408984-0.624132-0.735857-0.121553
Function 7: 0.076466-0.107136-0.246213-0.221705-0.391060-0.600276
Function 8: 0.123965-0.028657-0.123383-0.188126-0.900411-0.481502-0.091022-0.738327


C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 3 of parameter k1__k2__length_scale is close to the specified upper bound 8.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 4 of parameter k1__k2__length_scale is close to the specified upper bound 8.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 5 of parameter k1__k2__length_scale is close to the specified upper bound 8.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
C:\Users\R

In [1]:
import numpy as np

from pathlib import Path

from sklearn.gaussian_process import GaussianProcessRegressor

from sklearn.gaussian_process.kernels import ConstantKernel, Matern, WhiteKernel

from sklearn.preprocessing import StandardScaler



base = Path(r"C:\Users\Ramnath Lakshmanan\Downloads\Initial_data_points_starter (2)\initial_data")

DIMS = {1: 2, 2: 2, 3: 3, 4: 4, 5: 4, 6: 5, 7: 6, 8: 8}



W1X = {

    1: np.array([0.973822, 0.110674]),

    2: np.array([0.711216, 0.383626]),

    3: np.array([0.628013, 0.359224, 0.539509]),

    4: np.array([0.367191, 0.464182, 0.391040, 0.448820]),

    5: np.array([0.245752, 0.932264, 0.962162, 0.965058]),

    6: np.array([0.446820, 0.321047, 0.436584, 0.762321, 0.003307]),

    7: np.array([0.086332, 0.371883, 0.229157, 0.106524, 0.394874, 0.689739]),

    8: np.array([0.054501, 0.081016, 0.320114, 0.398440, 0.621808, 0.649205, 0.338643, 0.601487]),

}

W1Y = {1:-1.1814831817160104e-270,2:0.6525848308450871,3:-0.01927280144331694,

       4:-0.9174251874710708,5:2830.5996301272953,6:-0.41995556157505487,

       7:1.5145555220314069,8:9.7443661466821}

W2X = {

    1: np.array([0.520875, 0.334357]),

    2: np.array([0.701530, 0.047056]),

    3: np.array([0.879583, 0.006977, 0.001530]),

    4: np.array([0.421199, 0.400087, 0.337912, 0.436790]),

    5: np.array([0.116598, 0.999999, 0.999999, 0.999999]),

    6: np.array([0.425618, 0.196929, 0.846445, 0.972258, 0.141083]),

    7: np.array([0.000001, 0.312636, 0.151505, 0.026906, 0.361543, 0.731254]),

    8: np.array([0.102526, 0.102909, 0.008115, 0.245476, 0.999999, 0.523852, 0.248637, 0.499859]),

}

W2Y = {1:4.584141793749554e-13,2:0.6685300747584026,3:-0.18176416202017948,

       4:0.2852616086635966,5:4441.586450664394,6:-0.5975592068881134,

       7:1.0217094038661905,8:9.9177834946854}

W3X = {

    1: np.array([0.605824, 0.805874]),

    2: np.array([0.719374, 0.085613]),

    3: np.array([0.040729, 0.915411, 0.504944]),

    4: np.array([0.439246, 0.393165, 0.273345, 0.439606]),

    5: np.array([0.000001, 0.999999, 0.999999, 0.999999]),

    6: np.array([0.508487, 0.271802, 0.367910, 0.999999, 0.000001]),

    7: np.array([0.078771, 0.282330, 0.246212, 0.164149, 0.401057, 0.641158]),

    8: np.array([0.179705, 0.000001, 0.204942, 0.149746, 0.990645, 0.489105, 0.181870, 0.440792]),

}

W3Y = {1:5.221216935504337e-23,2:0.6685229247231236,3:-0.030942339134816123,

       4:-1.2975118969582797,5:4440.480873839292,6:-0.7811647047830718,

       7:1.9868235528281357,8:9.92646197977771}

W4X = {

    1: np.array([0.906435, 0.524773]),

    2: np.array([0.711541, 0.000001]),

    3: np.array([0.901391, 0.100657, 0.501946]),

    4: np.array([0.404491, 0.411487, 0.374128, 0.419096]),

    5: np.array([0.070905, 0.999999, 0.999999, 0.812090]),

    6: np.array([0.423685, 0.314205, 0.861819, 0.607621, 0.009157]),

    7: np.array([0.053889, 0.170531, 0.230571, 0.223537, 0.393017, 0.615148]),

    8: np.array([0.000001, 0.000001, 0.058234, 0.160975, 0.905106, 0.422655, 0.072201, 0.456271]),

}

W4Y = {1:-9.171561473040102e-63,2:0.718243930604028,3:-0.05350002457722131,

       4:0.6098506102572618,5:2690.3948413647527,6:-0.6049887641373398,

       7:2.2735115338126204,8:9.8956923171149}

W5X = {

    1: np.array([0.276310, 0.463966]),

    2: np.array([0.711701, 0.012157]),

    3: np.array([0.812590, 0.971252, 0.407756]),

    4: np.array([0.449239, 0.320819, 0.430682, 0.426303]),

    5: np.array([0.029536, 0.999999, 0.999999, 0.999999]),

    6: np.array([0.440177, 0.307928, 0.516852, 0.718082, 0.163715]),

    7: np.array([0.000001, 0.079068, 0.203789, 0.237496, 0.371176, 0.578991]),

    8: np.array([0.116622, 0.061321, 0.145257, 0.029806, 0.891611, 0.487268, 0.132540, 0.643656]),

}

W5Y = {1:-1.4588117677798244e-17,2:0.7233854034475573,3:-0.0348256634794126,

       4:-0.3587979590239736,5:4440.508119101403,6:-0.3636516136843698,

       7:2.113146751542243,8:9.9627878480899}

W6X = {

    1: np.array([0.147012, 0.207382]),

    2: np.array([0.711320, 0.022518]),

    3: np.array([0.042548, 0.018175, 0.535544]),

    4: np.array([0.417683, 0.392791, 0.389865, 0.415352]),

    5: np.array([0.072146, 0.999999, 0.999999, 0.999999]),

    6: np.array([0.483442, 0.406320, 0.551572, 0.728395, 0.141301]),

    7: np.array([0.002952, 0.137104, 0.221381, 0.265099, 0.399907, 0.616118]),

    8: np.array([0.024400, 0.068670, 0.123985, 0.088530, 0.958126, 0.519233, 0.251301, 0.734513]),

}

W6Y = {1:7.513182419081889e-87,2:0.5266505044433385,3:-0.10301859579872755,

       4:0.45948056013104877,5:4440.724001387989,6:-0.29074974600067394,

       7:2.1465849895947136,8:9.9581228253791}

W7X = {

    1: np.array([0.623326, 0.011857]),

    2: np.array([0.688509, 0.006758]),

    3: np.array([0.973709, 0.001813, 0.792312]),

    4: np.array([0.411661, 0.384885, 0.399073, 0.428315]),

    5: np.array([0.206499, 0.999999, 0.999999, 0.999999]),

    6: np.array([0.528613, 0.408984, 0.624132, 0.735857, 0.121553]),

    7: np.array([0.076466, 0.107136, 0.246213, 0.221705, 0.391060, 0.600276]),

    8: np.array([0.123965, 0.028657, 0.123383, 0.188126, 0.900411, 0.481502, 0.091022, 0.738327]),

}

W7Y = {1:2.1604429724905267e-146,2:0.5854363115719756,3:-0.1366850708077359,

       4:0.43813490537661126,5:4448.754616848498,6:-0.1623020733630079,

       7:2.3287657825776096,8:9.9514930826326}



ANCHOR = {1: W2X[1], 2: W5X[2], 3: W1X[3], 4: W4X[4],

          5: W7X[5], 6: W7X[6], 7: W7X[7], 8: W5X[8]}

POLICY = {

    1: dict(kappa=1.8, local=0.08, min_dist=0.025),

    2: dict(kappa=0.6, local=0.025, min_dist=0.008),

    3: dict(kappa=1.1, local=0.07, min_dist=0.03),

    4: dict(kappa=0.7, local=0.04, min_dist=0.018),

    5: dict(kappa=0.5, local=0.03, min_dist=0.015),

    6: dict(kappa=0.7, local=0.05, min_dist=0.02),

    7: dict(kappa=0.6, local=0.05, min_dist=0.02),

    8: dict(kappa=0.7, local=0.04, min_dist=0.035),

}



def fmt(x):

    x = np.clip(np.asarray(x, float).ravel(), 1e-6, 1 - 1e-6)

    return "-".join(f"{v:.6f}" for v in x)



queries = {}

for k in range(1, 9):

    X0 = np.load(base / f"function_{k}" / "initial_inputs.npy").astype(float)

    y0 = np.load(base / f"function_{k}" / "initial_outputs.npy").astype(float).ravel()

    X = np.vstack([X0, W1X[k], W2X[k], W3X[k], W4X[k], W5X[k], W6X[k], W7X[k]])

    y = np.concatenate([y0, [W1Y[k], W2Y[k], W3Y[k], W4Y[k], W5Y[k], W6Y[k], W7Y[k]]])

    d, pol, rng = DIMS[k], POLICY[k], np.random.default_rng(8000 + k)

    print(f"F{k} n={len(y)} best={y.max():.6g} last={y[-1]:.6g}")

    xsc = StandardScaler().fit(X)

    gp = GaussianProcessRegressor(

        kernel=(ConstantKernel(1.0, (1e-3, 1e3))

                * Matern(np.full(d, 0.3), (0.05, 8.0), nu=2.5)

                + WhiteKernel(1e-5, (1e-12, 1e-1))),

        normalize_y=True, n_restarts_optimizer=2, random_state=k)

    gp.fit(xsc.transform(X), y)

    cand = np.clip(ANCHOR[k] + rng.normal(0, pol["local"], size=(9000, d)), 1e-6, 1-1e-6)

    cand = np.vstack([cand, rng.uniform(1e-6, 1-1e-6, size=(2500, d))])

    mu, std = gp.predict(xsc.transform(cand), return_std=True)

    scores = mu + pol["kappa"] * std

    dist = np.min(np.linalg.norm(cand[:, None, :] - X[None, :, :], axis=2), axis=1)

    scores = np.where(dist < pol["min_dist"], -np.inf, scores)

    q = fmt(cand[int(np.argmax(scores))])

    queries[k] = q

    print("  SUBMIT:", q, "\n")



print("=" * 60)

print("WEEK 8 — PASTE INTO THE PORTAL")

print("=" * 60)

for k, q in queries.items():

    print(f"Function {k}: {q}")

F1 n=17 best=4.58414e-13 last=2.16044e-146
  SUBMIT: 0.724072-0.949508 

F2 n=17 best=0.723385 last=0.585436


C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified upper bound 8.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


  SUBMIT: 0.728329-0.006445 

F3 n=22 best=-0.0192728 last=-0.136685
  SUBMIT: 0.451949-0.969232-0.458722 

F4 n=37 best=0.609851 last=0.438135


C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 8.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


  SUBMIT: 0.404607-0.397025-0.403666-0.417850 

F5 n=27 best=4448.75 last=4448.75
  SUBMIT: 0.301402-0.999999-0.999999-0.999999 

F6 n=27 best=-0.162302 last=-0.162302


C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified upper bound 8.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\gaussian_process\kernels.py:445: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-12. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


  SUBMIT: 0.538213-0.415090-0.660675-0.743991-0.110951 

F7 n=37 best=2.32877 last=2.32877


C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 2 of parameter k1__k2__length_scale is close to the specified upper bound 8.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


  SUBMIT: 0.119224-0.110793-0.167022-0.243798-0.404717-0.589881 

F8 n=47 best=9.96279 last=9.95149
  SUBMIT: 0.112099-0.000001-0.112888-0.055898-0.872203-0.396747-0.171766-0.681514 

WEEK 8 — PASTE INTO THE PORTAL
Function 1: 0.724072-0.949508
Function 2: 0.728329-0.006445
Function 3: 0.451949-0.969232-0.458722
Function 4: 0.404607-0.397025-0.403666-0.417850
Function 5: 0.301402-0.999999-0.999999-0.999999
Function 6: 0.538213-0.415090-0.660675-0.743991-0.110951
Function 7: 0.119224-0.110793-0.167022-0.243798-0.404717-0.589881
Function 8: 0.112099-0.000001-0.112888-0.055898-0.872203-0.396747-0.171766-0.681514


C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 3 of parameter k1__k2__length_scale is close to the specified upper bound 8.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 4 of parameter k1__k2__length_scale is close to the specified upper bound 8.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 5 of parameter k1__k2__length_scale is close to the specified upper bound 8.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
C:\Users\R

In [1]:
import numpy as np

from pathlib import Path

from sklearn.gaussian_process import GaussianProcessRegressor

from sklearn.gaussian_process.kernels import ConstantKernel, Matern, WhiteKernel

from sklearn.preprocessing import StandardScaler



base = Path(r"C:\Users\Ramnath Lakshmanan\Downloads\Initial_data_points_starter (2)\initial_data")

DIMS = {1: 2, 2: 2, 3: 3, 4: 4, 5: 4, 6: 5, 7: 6, 8: 8}



W1X = {

    1: np.array([0.973822, 0.110674]), 2: np.array([0.711216, 0.383626]),

    3: np.array([0.628013, 0.359224, 0.539509]),

    4: np.array([0.367191, 0.464182, 0.391040, 0.448820]),

    5: np.array([0.245752, 0.932264, 0.962162, 0.965058]),

    6: np.array([0.446820, 0.321047, 0.436584, 0.762321, 0.003307]),

    7: np.array([0.086332, 0.371883, 0.229157, 0.106524, 0.394874, 0.689739]),

    8: np.array([0.054501, 0.081016, 0.320114, 0.398440, 0.621808, 0.649205, 0.338643, 0.601487]),

}

W1Y = {1:-1.1814831817160104e-270,2:0.6525848308450871,3:-0.01927280144331694,

       4:-0.9174251874710708,5:2830.5996301272953,6:-0.41995556157505487,

       7:1.5145555220314069,8:9.7443661466821}

W2X = {

    1: np.array([0.520875, 0.334357]), 2: np.array([0.701530, 0.047056]),

    3: np.array([0.879583, 0.006977, 0.001530]),

    4: np.array([0.421199, 0.400087, 0.337912, 0.436790]),

    5: np.array([0.116598, 0.999999, 0.999999, 0.999999]),

    6: np.array([0.425618, 0.196929, 0.846445, 0.972258, 0.141083]),

    7: np.array([0.000001, 0.312636, 0.151505, 0.026906, 0.361543, 0.731254]),

    8: np.array([0.102526, 0.102909, 0.008115, 0.245476, 0.999999, 0.523852, 0.248637, 0.499859]),

}

W2Y = {1:4.584141793749554e-13,2:0.6685300747584026,3:-0.18176416202017948,

       4:0.2852616086635966,5:4441.586450664394,6:-0.5975592068881134,

       7:1.0217094038661905,8:9.9177834946854}

W3X = {

    1: np.array([0.605824, 0.805874]), 2: np.array([0.719374, 0.085613]),

    3: np.array([0.040729, 0.915411, 0.504944]),

    4: np.array([0.439246, 0.393165, 0.273345, 0.439606]),

    5: np.array([0.000001, 0.999999, 0.999999, 0.999999]),

    6: np.array([0.508487, 0.271802, 0.367910, 0.999999, 0.000001]),

    7: np.array([0.078771, 0.282330, 0.246212, 0.164149, 0.401057, 0.641158]),

    8: np.array([0.179705, 0.000001, 0.204942, 0.149746, 0.990645, 0.489105, 0.181870, 0.440792]),

}

W3Y = {1:5.221216935504337e-23,2:0.6685229247231236,3:-0.030942339134816123,

       4:-1.2975118969582797,5:4440.480873839292,6:-0.7811647047830718,

       7:1.9868235528281357,8:9.92646197977771}

W4X = {

    1: np.array([0.906435, 0.524773]), 2: np.array([0.711541, 0.000001]),

    3: np.array([0.901391, 0.100657, 0.501946]),

    4: np.array([0.404491, 0.411487, 0.374128, 0.419096]),

    5: np.array([0.070905, 0.999999, 0.999999, 0.812090]),

    6: np.array([0.423685, 0.314205, 0.861819, 0.607621, 0.009157]),

    7: np.array([0.053889, 0.170531, 0.230571, 0.223537, 0.393017, 0.615148]),

    8: np.array([0.000001, 0.000001, 0.058234, 0.160975, 0.905106, 0.422655, 0.072201, 0.456271]),

}

W4Y = {1:-9.171561473040102e-63,2:0.718243930604028,3:-0.05350002457722131,

       4:0.6098506102572618,5:2690.3948413647527,6:-0.6049887641373398,

       7:2.2735115338126204,8:9.8956923171149}

W5X = {

    1: np.array([0.276310, 0.463966]), 2: np.array([0.711701, 0.012157]),

    3: np.array([0.812590, 0.971252, 0.407756]),

    4: np.array([0.449239, 0.320819, 0.430682, 0.426303]),

    5: np.array([0.029536, 0.999999, 0.999999, 0.999999]),

    6: np.array([0.440177, 0.307928, 0.516852, 0.718082, 0.163715]),

    7: np.array([0.000001, 0.079068, 0.203789, 0.237496, 0.371176, 0.578991]),

    8: np.array([0.116622, 0.061321, 0.145257, 0.029806, 0.891611, 0.487268, 0.132540, 0.643656]),

}

W5Y = {1:-1.4588117677798244e-17,2:0.7233854034475573,3:-0.0348256634794126,

       4:-0.3587979590239736,5:4440.508119101403,6:-0.3636516136843698,

       7:2.113146751542243,8:9.9627878480899}

W6X = {

    1: np.array([0.147012, 0.207382]), 2: np.array([0.711320, 0.022518]),

    3: np.array([0.042548, 0.018175, 0.535544]),

    4: np.array([0.417683, 0.392791, 0.389865, 0.415352]),

    5: np.array([0.072146, 0.999999, 0.999999, 0.999999]),

    6: np.array([0.483442, 0.406320, 0.551572, 0.728395, 0.141301]),

    7: np.array([0.002952, 0.137104, 0.221381, 0.265099, 0.399907, 0.616118]),

    8: np.array([0.024400, 0.068670, 0.123985, 0.088530, 0.958126, 0.519233, 0.251301, 0.734513]),

}

W6Y = {1:7.513182419081889e-87,2:0.5266505044433385,3:-0.10301859579872755,

       4:0.45948056013104877,5:4440.724001387989,6:-0.29074974600067394,

       7:2.1465849895947136,8:9.9581228253791}

W7X = {

    1: np.array([0.623326, 0.011857]), 2: np.array([0.688509, 0.006758]),

    3: np.array([0.973709, 0.001813, 0.792312]),

    4: np.array([0.411661, 0.384885, 0.399073, 0.428315]),

    5: np.array([0.206499, 0.999999, 0.999999, 0.999999]),

    6: np.array([0.528613, 0.408984, 0.624132, 0.735857, 0.121553]),

    7: np.array([0.076466, 0.107136, 0.246213, 0.221705, 0.391060, 0.600276]),

    8: np.array([0.123965, 0.028657, 0.123383, 0.188126, 0.900411, 0.481502, 0.091022, 0.738327]),

}

W7Y = {1:2.1604429724905267e-146,2:0.5854363115719756,3:-0.1366850708077359,

       4:0.43813490537661126,5:4448.754616848498,6:-0.1623020733630079,

       7:2.3287657825776096,8:9.9514930826326}

W8X = {

    1: np.array([0.724072, 0.949508]), 2: np.array([0.728329, 0.006445]),

    3: np.array([0.451949, 0.969232, 0.458722]),

    4: np.array([0.404607, 0.397025, 0.403666, 0.417850]),

    5: np.array([0.301402, 0.999999, 0.999999, 0.999999]),

    6: np.array([0.538213, 0.415090, 0.660675, 0.743991, 0.110951]),

    7: np.array([0.119224, 0.110793, 0.167022, 0.243798, 0.404717, 0.589881]),

    8: np.array([0.112099, 0.000001, 0.112888, 0.055898, 0.872203, 0.396747, 0.171766, 0.681514]),

}

W8Y = {1:-1.7695749378252557e-78,2:0.5691554534436665,3:-0.03431158030376151,

       4:0.4792797063547316,5:4474.337323581212,6:-0.18658854384879187,

       7:2.186293628123943,8:9.9519472910159}



ANCHOR = {1: W2X[1], 2: W5X[2], 3: W1X[3], 4: W4X[4],

          5: W8X[5], 6: W7X[6], 7: W7X[7], 8: W5X[8]}

POLICY = {

    1: dict(kappa=1.8, local=0.08, min_dist=0.025),

    2: dict(kappa=0.55, local=0.02, min_dist=0.008),

    3: dict(kappa=1.1, local=0.07, min_dist=0.03),

    4: dict(kappa=0.7, local=0.04, min_dist=0.018),

    5: dict(kappa=0.45, local=0.025, min_dist=0.012),

    6: dict(kappa=0.7, local=0.045, min_dist=0.02),

    7: dict(kappa=0.6, local=0.045, min_dist=0.02),

    8: dict(kappa=0.7, local=0.04, min_dist=0.035),

}



def fmt(x):

    x = np.clip(np.asarray(x, float).ravel(), 1e-6, 1-1e-6)

    return "-".join(f"{v:.6f}" for v in x)



queries = {}

for k in range(1, 9):

    X0 = np.load(base / f"function_{k}" / "initial_inputs.npy").astype(float)

    y0 = np.load(base / f"function_{k}" / "initial_outputs.npy").astype(float).ravel()

    X = np.vstack([X0, W1X[k], W2X[k], W3X[k], W4X[k], W5X[k], W6X[k], W7X[k], W8X[k]])

    y = np.concatenate([y0, [W1Y[k], W2Y[k], W3Y[k], W4Y[k], W5Y[k], W6Y[k], W7Y[k], W8Y[k]]])

    d, pol, rng = DIMS[k], POLICY[k], np.random.default_rng(9000 + k)

    print(f"F{k} n={len(y)} best={y.max():.6g} last={y[-1]:.6g}")

    xsc = StandardScaler().fit(X)

    gp = GaussianProcessRegressor(

        kernel=(ConstantKernel(1.0, (1e-3, 1e3))

                * Matern(np.full(d, 0.3), (0.05, 8.0), nu=2.5)

                + WhiteKernel(1e-5, (1e-12, 1e-1))),

        normalize_y=True, n_restarts_optimizer=2, random_state=k)

    gp.fit(xsc.transform(X), y)

    cand = np.clip(ANCHOR[k] + rng.normal(0, pol["local"], size=(9000, d)), 1e-6, 1-1e-6)

    cand = np.vstack([cand, rng.uniform(1e-6, 1-1e-6, size=(2500, d))])

    mu, std = gp.predict(xsc.transform(cand), return_std=True)

    scores = mu + pol["kappa"] * std

    dist = np.min(np.linalg.norm(cand[:, None, :] - X[None, :, :], axis=2), axis=1)

    scores = np.where(dist < pol["min_dist"], -np.inf, scores)

    q = fmt(cand[int(np.argmax(scores))])

    queries[k] = q

    print("  SUBMIT:", q, "\n")



print("=" * 60)

print("WEEK 9 — PASTE INTO THE PORTAL")

print("=" * 60)

for k, q in queries.items():

    print(f"Function {k}: {q}")

F1 n=18 best=4.58414e-13 last=-1.76957e-78
  SUBMIT: 0.032731-0.999991 

F2 n=18 best=0.723385 last=0.569155
  SUBMIT: 0.705749-0.006184 

F3 n=23 best=-0.0192728 last=-0.0343116


C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified upper bound 8.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified upper bound 8.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


  SUBMIT: 0.852022-0.624763-0.509852 

F4 n=38 best=0.609851 last=0.47928
  SUBMIT: 0.401861-0.377528-0.411745-0.423788 

F5 n=28 best=4474.34 last=4474.34


C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified upper bound 8.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\gaussian_process\kernels.py:445: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-12. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


  SUBMIT: 0.376936-0.999999-0.999999-0.999999 

F6 n=28 best=-0.162302 last=-0.186589
  SUBMIT: 0.558753-0.355800-0.638855-0.725691-0.108270 

F7 n=38 best=2.32877 last=2.18629


C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 2 of parameter k1__k2__length_scale is close to the specified upper bound 8.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


  SUBMIT: 0.063449-0.102659-0.338015-0.219698-0.430000-0.636966 

F8 n=48 best=9.96279 last=9.95195
  SUBMIT: 0.090049-0.000001-0.094558-0.058201-0.964454-0.566320-0.154497-0.712821 

WEEK 9 — PASTE INTO THE PORTAL
Function 1: 0.032731-0.999991
Function 2: 0.705749-0.006184
Function 3: 0.852022-0.624763-0.509852
Function 4: 0.401861-0.377528-0.411745-0.423788
Function 5: 0.376936-0.999999-0.999999-0.999999
Function 6: 0.558753-0.355800-0.638855-0.725691-0.108270
Function 7: 0.063449-0.102659-0.338015-0.219698-0.430000-0.636966
Function 8: 0.090049-0.000001-0.094558-0.058201-0.964454-0.566320-0.154497-0.712821


C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 3 of parameter k1__k2__length_scale is close to the specified upper bound 8.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 4 of parameter k1__k2__length_scale is close to the specified upper bound 8.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 5 of parameter k1__k2__length_scale is close to the specified upper bound 8.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
C:\Users\R

In [1]:
import numpy as np

from pathlib import Path

from sklearn.gaussian_process import GaussianProcessRegressor

from sklearn.gaussian_process.kernels import ConstantKernel, Matern, WhiteKernel

from sklearn.preprocessing import StandardScaler



base = Path(r"C:\Users\Ramnath Lakshmanan\Downloads\Initial_data_points_starter (2)\initial_data")

DIMS = {1: 2, 2: 2, 3: 3, 4: 4, 5: 4, 6: 5, 7: 6, 8: 8}



W1X = {

    1: np.array([0.973822, 0.110674]), 2: np.array([0.711216, 0.383626]),

    3: np.array([0.628013, 0.359224, 0.539509]),

    4: np.array([0.367191, 0.464182, 0.391040, 0.448820]),

    5: np.array([0.245752, 0.932264, 0.962162, 0.965058]),

    6: np.array([0.446820, 0.321047, 0.436584, 0.762321, 0.003307]),

    7: np.array([0.086332, 0.371883, 0.229157, 0.106524, 0.394874, 0.689739]),

    8: np.array([0.054501, 0.081016, 0.320114, 0.398440, 0.621808, 0.649205, 0.338643, 0.601487]),

}

W1Y = {1:-1.1814831817160104e-270,2:0.6525848308450871,3:-0.01927280144331694,

       4:-0.9174251874710708,5:2830.5996301272953,6:-0.41995556157505487,

       7:1.5145555220314069,8:9.7443661466821}

W2X = {

    1: np.array([0.520875, 0.334357]), 2: np.array([0.701530, 0.047056]),

    3: np.array([0.879583, 0.006977, 0.001530]),

    4: np.array([0.421199, 0.400087, 0.337912, 0.436790]),

    5: np.array([0.116598, 0.999999, 0.999999, 0.999999]),

    6: np.array([0.425618, 0.196929, 0.846445, 0.972258, 0.141083]),

    7: np.array([0.000001, 0.312636, 0.151505, 0.026906, 0.361543, 0.731254]),

    8: np.array([0.102526, 0.102909, 0.008115, 0.245476, 0.999999, 0.523852, 0.248637, 0.499859]),

}

W2Y = {1:4.584141793749554e-13,2:0.6685300747584026,3:-0.18176416202017948,

       4:0.2852616086635966,5:4441.586450664394,6:-0.5975592068881134,

       7:1.0217094038661905,8:9.9177834946854}

W3X = {

    1: np.array([0.605824, 0.805874]), 2: np.array([0.719374, 0.085613]),

    3: np.array([0.040729, 0.915411, 0.504944]),

    4: np.array([0.439246, 0.393165, 0.273345, 0.439606]),

    5: np.array([0.000001, 0.999999, 0.999999, 0.999999]),

    6: np.array([0.508487, 0.271802, 0.367910, 0.999999, 0.000001]),

    7: np.array([0.078771, 0.282330, 0.246212, 0.164149, 0.401057, 0.641158]),

    8: np.array([0.179705, 0.000001, 0.204942, 0.149746, 0.990645, 0.489105, 0.181870, 0.440792]),

}

W3Y = {1:5.221216935504337e-23,2:0.6685229247231236,3:-0.030942339134816123,

       4:-1.2975118969582797,5:4440.480873839292,6:-0.7811647047830718,

       7:1.9868235528281357,8:9.92646197977771}

W4X = {

    1: np.array([0.906435, 0.524773]), 2: np.array([0.711541, 0.000001]),

    3: np.array([0.901391, 0.100657, 0.501946]),

    4: np.array([0.404491, 0.411487, 0.374128, 0.419096]),

    5: np.array([0.070905, 0.999999, 0.999999, 0.812090]),

    6: np.array([0.423685, 0.314205, 0.861819, 0.607621, 0.009157]),

    7: np.array([0.053889, 0.170531, 0.230571, 0.223537, 0.393017, 0.615148]),

    8: np.array([0.000001, 0.000001, 0.058234, 0.160975, 0.905106, 0.422655, 0.072201, 0.456271]),

}

W4Y = {1:-9.171561473040102e-63,2:0.718243930604028,3:-0.05350002457722131,

       4:0.6098506102572618,5:2690.3948413647527,6:-0.6049887641373398,

       7:2.2735115338126204,8:9.8956923171149}

W5X = {

    1: np.array([0.276310, 0.463966]), 2: np.array([0.711701, 0.012157]),

    3: np.array([0.812590, 0.971252, 0.407756]),

    4: np.array([0.449239, 0.320819, 0.430682, 0.426303]),

    5: np.array([0.029536, 0.999999, 0.999999, 0.999999]),

    6: np.array([0.440177, 0.307928, 0.516852, 0.718082, 0.163715]),

    7: np.array([0.000001, 0.079068, 0.203789, 0.237496, 0.371176, 0.578991]),

    8: np.array([0.116622, 0.061321, 0.145257, 0.029806, 0.891611, 0.487268, 0.132540, 0.643656]),

}

W5Y = {1:-1.4588117677798244e-17,2:0.7233854034475573,3:-0.0348256634794126,

       4:-0.3587979590239736,5:4440.508119101403,6:-0.3636516136843698,

       7:2.113146751542243,8:9.9627878480899}

W6X = {

    1: np.array([0.147012, 0.207382]), 2: np.array([0.711320, 0.022518]),

    3: np.array([0.042548, 0.018175, 0.535544]),

    4: np.array([0.417683, 0.392791, 0.389865, 0.415352]),

    5: np.array([0.072146, 0.999999, 0.999999, 0.999999]),

    6: np.array([0.483442, 0.406320, 0.551572, 0.728395, 0.141301]),

    7: np.array([0.002952, 0.137104, 0.221381, 0.265099, 0.399907, 0.616118]),

    8: np.array([0.024400, 0.068670, 0.123985, 0.088530, 0.958126, 0.519233, 0.251301, 0.734513]),

}

W6Y = {1:7.513182419081889e-87,2:0.5266505044433385,3:-0.10301859579872755,

       4:0.45948056013104877,5:4440.724001387989,6:-0.29074974600067394,

       7:2.1465849895947136,8:9.9581228253791}

W7X = {

    1: np.array([0.623326, 0.011857]), 2: np.array([0.688509, 0.006758]),

    3: np.array([0.973709, 0.001813, 0.792312]),

    4: np.array([0.411661, 0.384885, 0.399073, 0.428315]),

    5: np.array([0.206499, 0.999999, 0.999999, 0.999999]),

    6: np.array([0.528613, 0.408984, 0.624132, 0.735857, 0.121553]),

    7: np.array([0.076466, 0.107136, 0.246213, 0.221705, 0.391060, 0.600276]),

    8: np.array([0.123965, 0.028657, 0.123383, 0.188126, 0.900411, 0.481502, 0.091022, 0.738327]),

}

W7Y = {1:2.1604429724905267e-146,2:0.5854363115719756,3:-0.1366850708077359,

       4:0.43813490537661126,5:4448.754616848498,6:-0.1623020733630079,

       7:2.3287657825776096,8:9.9514930826326}

W8X = {

    1: np.array([0.724072, 0.949508]), 2: np.array([0.728329, 0.006445]),

    3: np.array([0.451949, 0.969232, 0.458722]),

    4: np.array([0.404607, 0.397025, 0.403666, 0.417850]),

    5: np.array([0.301402, 0.999999, 0.999999, 0.999999]),

    6: np.array([0.538213, 0.415090, 0.660675, 0.743991, 0.110951]),

    7: np.array([0.119224, 0.110793, 0.167022, 0.243798, 0.404717, 0.589881]),

    8: np.array([0.112099, 0.000001, 0.112888, 0.055898, 0.872203, 0.396747, 0.171766, 0.681514]),

}

W8Y = {1:-1.7695749378252557e-78,2:0.5691554534436665,3:-0.03431158030376151,

       4:0.4792797063547316,5:4474.337323581212,6:-0.18658854384879187,

       7:2.186293628123943,8:9.9519472910159}

W9X = {

    1: np.array([0.032731, 0.999991]), 2: np.array([0.705749, 0.006184]),

    3: np.array([0.852022, 0.624763, 0.509852]),

    4: np.array([0.401861, 0.377528, 0.411745, 0.423788]),

    5: np.array([0.376936, 0.999999, 0.999999, 0.999999]),

    6: np.array([0.558753, 0.355800, 0.638855, 0.725691, 0.108270]),

    7: np.array([0.063449, 0.102659, 0.338015, 0.219698, 0.430000, 0.636966]),

    8: np.array([0.090049, 0.000001, 0.094558, 0.058201, 0.964454, 0.566320, 0.154497, 0.712821]),

}

W9Y = {1:0.0,2:0.5641261834450958,3:-0.006194559125586974,

       4:0.5477069467900759,5:4519.925222262125,6:-0.2780636435928158,

       7:2.2506650591726416,8:9.9417719874239}



ANCHOR = {1: W2X[1], 2: W5X[2], 3: W9X[3], 4: W4X[4],

          5: W9X[5], 6: W7X[6], 7: W7X[7], 8: W5X[8]}

POLICY = {

    1: dict(kappa=1.8, local=0.08, min_dist=0.025),

    2: dict(kappa=0.5, local=0.018, min_dist=0.006),

    3: dict(kappa=0.8, local=0.05, min_dist=0.02),

    4: dict(kappa=0.65, local=0.035, min_dist=0.015),

    5: dict(kappa=0.4, local=0.03, min_dist=0.012),

    6: dict(kappa=0.7, local=0.045, min_dist=0.02),

    7: dict(kappa=0.6, local=0.04, min_dist=0.02),

    8: dict(kappa=0.7, local=0.04, min_dist=0.03),

}



def fmt(x):

    x = np.clip(np.asarray(x, float).ravel(), 1e-6, 1-1e-6)

    return "-".join(f"{v:.6f}" for v in x)



queries = {}

for k in range(1, 9):

    X0 = np.load(base / f"function_{k}" / "initial_inputs.npy").astype(float)

    y0 = np.load(base / f"function_{k}" / "initial_outputs.npy").astype(float).ravel()

    X = np.vstack([X0, W1X[k], W2X[k], W3X[k], W4X[k], W5X[k], W6X[k], W7X[k], W8X[k], W9X[k]])

    y = np.concatenate([y0, [W1Y[k], W2Y[k], W3Y[k], W4Y[k], W5Y[k], W6Y[k], W7Y[k], W8Y[k], W9Y[k]]])

    d, pol, rng = DIMS[k], POLICY[k], np.random.default_rng(10000 + k)

    print(f"F{k} n={len(y)} best={y.max():.6g} last={y[-1]:.6g}")

    xsc = StandardScaler().fit(X)

    gp = GaussianProcessRegressor(

        kernel=(ConstantKernel(1.0, (1e-3, 1e3))

                * Matern(np.full(d, 0.3), (0.05, 8.0), nu=2.5)

                + WhiteKernel(1e-5, (1e-12, 1e-1))),

        normalize_y=True, n_restarts_optimizer=2, random_state=k)

    gp.fit(xsc.transform(X), y)

    cand = np.clip(ANCHOR[k] + rng.normal(0, pol["local"], size=(9000, d)), 1e-6, 1-1e-6)

    cand = np.vstack([cand, rng.uniform(1e-6, 1-1e-6, size=(2000, d))])

    mu, std = gp.predict(xsc.transform(cand), return_std=True)

    scores = mu + pol["kappa"] * std

    dist = np.min(np.linalg.norm(cand[:, None, :] - X[None, :, :], axis=2), axis=1)

    scores = np.where(dist < pol["min_dist"], -np.inf, scores)

    q = fmt(cand[int(np.argmax(scores))])

    queries[k] = q

    print("  SUBMIT:", q, "\n")



print("=" * 60)

print("WEEK 10 — PASTE INTO THE PORTAL")

print("=" * 60)

for k, q in queries.items():

    print(f"Function {k}: {q}")

F1 n=19 best=4.58414e-13 last=0
  SUBMIT: 0.098904-0.300459 

F2 n=19 best=0.723385 last=0.564126
  SUBMIT: 0.715876-0.005989 



C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified upper bound 8.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


F3 n=24 best=-0.00619456 last=-0.00619456
  SUBMIT: 0.196663-0.981783-0.000674 

F4 n=39 best=0.609851 last=0.547707
  SUBMIT: 0.409992-0.382531-0.400832-0.410710 

F5 n=29 best=4519.93 last=4519.93


C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\gaussian_process\_gpr.py:667: ConvergenceWarning: lbfgs failed to converge after 33 iteration(s) (status=2):
ABNORMAL: 

You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  _check_optimize_result("lbfgs", opt_res)
C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified upper bound 8.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\gaussian_process\kernels.py:445: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-12. Decreasing the bound and calling fit again may find a better value.
  warni

  SUBMIT: 0.468240-0.999999-0.999999-0.999999 

F6 n=29 best=-0.162302 last=-0.278064


C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 2 of parameter k1__k2__length_scale is close to the specified upper bound 8.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


  SUBMIT: 0.525876-0.321840-0.718922-0.725188-0.164633 

F7 n=39 best=2.32877 last=2.25067


C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 2 of parameter k1__k2__length_scale is close to the specified upper bound 8.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


  SUBMIT: 0.064058-0.024164-0.298611-0.247300-0.389429-0.590442 

F8 n=49 best=9.96279 last=9.94177
  SUBMIT: 0.088168-0.185331-0.145690-0.113714-0.786327-0.491018-0.182694-0.611969 

WEEK 10 — PASTE INTO THE PORTAL
Function 1: 0.098904-0.300459
Function 2: 0.715876-0.005989
Function 3: 0.196663-0.981783-0.000674
Function 4: 0.409992-0.382531-0.400832-0.410710
Function 5: 0.468240-0.999999-0.999999-0.999999
Function 6: 0.525876-0.321840-0.718922-0.725188-0.164633
Function 7: 0.064058-0.024164-0.298611-0.247300-0.389429-0.590442
Function 8: 0.088168-0.185331-0.145690-0.113714-0.786327-0.491018-0.182694-0.611969


C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 3 of parameter k1__k2__length_scale is close to the specified upper bound 8.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 4 of parameter k1__k2__length_scale is close to the specified upper bound 8.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 5 of parameter k1__k2__length_scale is close to the specified upper bound 8.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
C:\Users\R

In [1]:
import numpy as np

from pathlib import Path

from sklearn.gaussian_process import GaussianProcessRegressor

from sklearn.gaussian_process.kernels import ConstantKernel, Matern, WhiteKernel

from sklearn.preprocessing import StandardScaler



base = Path(r"C:\Users\Ramnath Lakshmanan\Downloads\Initial_data_points_starter (2)\initial_data")

DIMS = {1: 2, 2: 2, 3: 3, 4: 4, 5: 4, 6: 5, 7: 6, 8: 8}



W1X = {

    1: np.array([0.973822, 0.110674]), 2: np.array([0.711216, 0.383626]),

    3: np.array([0.628013, 0.359224, 0.539509]),

    4: np.array([0.367191, 0.464182, 0.391040, 0.448820]),

    5: np.array([0.245752, 0.932264, 0.962162, 0.965058]),

    6: np.array([0.446820, 0.321047, 0.436584, 0.762321, 0.003307]),

    7: np.array([0.086332, 0.371883, 0.229157, 0.106524, 0.394874, 0.689739]),

    8: np.array([0.054501, 0.081016, 0.320114, 0.398440, 0.621808, 0.649205, 0.338643, 0.601487]),

}

W1Y = {1:-1.1814831817160104e-270,2:0.6525848308450871,3:-0.01927280144331694,

       4:-0.9174251874710708,5:2830.5996301272953,6:-0.41995556157505487,

       7:1.5145555220314069,8:9.7443661466821}

W2X = {

    1: np.array([0.520875, 0.334357]), 2: np.array([0.701530, 0.047056]),

    3: np.array([0.879583, 0.006977, 0.001530]),

    4: np.array([0.421199, 0.400087, 0.337912, 0.436790]),

    5: np.array([0.116598, 0.999999, 0.999999, 0.999999]),

    6: np.array([0.425618, 0.196929, 0.846445, 0.972258, 0.141083]),

    7: np.array([0.000001, 0.312636, 0.151505, 0.026906, 0.361543, 0.731254]),

    8: np.array([0.102526, 0.102909, 0.008115, 0.245476, 0.999999, 0.523852, 0.248637, 0.499859]),

}

W2Y = {1:4.584141793749554e-13,2:0.6685300747584026,3:-0.18176416202017948,

       4:0.2852616086635966,5:4441.586450664394,6:-0.5975592068881134,

       7:1.0217094038661905,8:9.9177834946854}

W3X = {

    1: np.array([0.605824, 0.805874]), 2: np.array([0.719374, 0.085613]),

    3: np.array([0.040729, 0.915411, 0.504944]),

    4: np.array([0.439246, 0.393165, 0.273345, 0.439606]),

    5: np.array([0.000001, 0.999999, 0.999999, 0.999999]),

    6: np.array([0.508487, 0.271802, 0.367910, 0.999999, 0.000001]),

    7: np.array([0.078771, 0.282330, 0.246212, 0.164149, 0.401057, 0.641158]),

    8: np.array([0.179705, 0.000001, 0.204942, 0.149746, 0.990645, 0.489105, 0.181870, 0.440792]),

}

W3Y = {1:5.221216935504337e-23,2:0.6685229247231236,3:-0.030942339134816123,

       4:-1.2975118969582797,5:4440.480873839292,6:-0.7811647047830718,

       7:1.9868235528281357,8:9.92646197977771}

W4X = {

    1: np.array([0.906435, 0.524773]), 2: np.array([0.711541, 0.000001]),

    3: np.array([0.901391, 0.100657, 0.501946]),

    4: np.array([0.404491, 0.411487, 0.374128, 0.419096]),

    5: np.array([0.070905, 0.999999, 0.999999, 0.812090]),

    6: np.array([0.423685, 0.314205, 0.861819, 0.607621, 0.009157]),

    7: np.array([0.053889, 0.170531, 0.230571, 0.223537, 0.393017, 0.615148]),

    8: np.array([0.000001, 0.000001, 0.058234, 0.160975, 0.905106, 0.422655, 0.072201, 0.456271]),

}

W4Y = {1:-9.171561473040102e-63,2:0.718243930604028,3:-0.05350002457722131,

       4:0.6098506102572618,5:2690.3948413647527,6:-0.6049887641373398,

       7:2.2735115338126204,8:9.8956923171149}

W5X = {

    1: np.array([0.276310, 0.463966]), 2: np.array([0.711701, 0.012157]),

    3: np.array([0.812590, 0.971252, 0.407756]),

    4: np.array([0.449239, 0.320819, 0.430682, 0.426303]),

    5: np.array([0.029536, 0.999999, 0.999999, 0.999999]),

    6: np.array([0.440177, 0.307928, 0.516852, 0.718082, 0.163715]),

    7: np.array([0.000001, 0.079068, 0.203789, 0.237496, 0.371176, 0.578991]),

    8: np.array([0.116622, 0.061321, 0.145257, 0.029806, 0.891611, 0.487268, 0.132540, 0.643656]),

}

W5Y = {1:-1.4588117677798244e-17,2:0.7233854034475573,3:-0.0348256634794126,

       4:-0.3587979590239736,5:4440.508119101403,6:-0.3636516136843698,

       7:2.113146751542243,8:9.9627878480899}

W6X = {

    1: np.array([0.147012, 0.207382]), 2: np.array([0.711320, 0.022518]),

    3: np.array([0.042548, 0.018175, 0.535544]),

    4: np.array([0.417683, 0.392791, 0.389865, 0.415352]),

    5: np.array([0.072146, 0.999999, 0.999999, 0.999999]),

    6: np.array([0.483442, 0.406320, 0.551572, 0.728395, 0.141301]),

    7: np.array([0.002952, 0.137104, 0.221381, 0.265099, 0.399907, 0.616118]),

    8: np.array([0.024400, 0.068670, 0.123985, 0.088530, 0.958126, 0.519233, 0.251301, 0.734513]),

}

W6Y = {1:7.513182419081889e-87,2:0.5266505044433385,3:-0.10301859579872755,

       4:0.45948056013104877,5:4440.724001387989,6:-0.29074974600067394,

       7:2.1465849895947136,8:9.9581228253791}

W7X = {

    1: np.array([0.623326, 0.011857]), 2: np.array([0.688509, 0.006758]),

    3: np.array([0.973709, 0.001813, 0.792312]),

    4: np.array([0.411661, 0.384885, 0.399073, 0.428315]),

    5: np.array([0.206499, 0.999999, 0.999999, 0.999999]),

    6: np.array([0.528613, 0.408984, 0.624132, 0.735857, 0.121553]),

    7: np.array([0.076466, 0.107136, 0.246213, 0.221705, 0.391060, 0.600276]),

    8: np.array([0.123965, 0.028657, 0.123383, 0.188126, 0.900411, 0.481502, 0.091022, 0.738327]),

}

W7Y = {1:2.1604429724905267e-146,2:0.5854363115719756,3:-0.1366850708077359,

       4:0.43813490537661126,5:4448.754616848498,6:-0.1623020733630079,

       7:2.3287657825776096,8:9.9514930826326}

W8X = {

    1: np.array([0.724072, 0.949508]), 2: np.array([0.728329, 0.006445]),

    3: np.array([0.451949, 0.969232, 0.458722]),

    4: np.array([0.404607, 0.397025, 0.403666, 0.417850]),

    5: np.array([0.301402, 0.999999, 0.999999, 0.999999]),

    6: np.array([0.538213, 0.415090, 0.660675, 0.743991, 0.110951]),

    7: np.array([0.119224, 0.110793, 0.167022, 0.243798, 0.404717, 0.589881]),

    8: np.array([0.112099, 0.000001, 0.112888, 0.055898, 0.872203, 0.396747, 0.171766, 0.681514]),

}

W8Y = {1:-1.7695749378252557e-78,2:0.5691554534436665,3:-0.03431158030376151,

       4:0.4792797063547316,5:4474.337323581212,6:-0.18658854384879187,

       7:2.186293628123943,8:9.9519472910159}

W9X = {

    1: np.array([0.032731, 0.999991]), 2: np.array([0.705749, 0.006184]),

    3: np.array([0.852022, 0.624763, 0.509852]),

    4: np.array([0.401861, 0.377528, 0.411745, 0.423788]),

    5: np.array([0.376936, 0.999999, 0.999999, 0.999999]),

    6: np.array([0.558753, 0.355800, 0.638855, 0.725691, 0.108270]),

    7: np.array([0.063449, 0.102659, 0.338015, 0.219698, 0.430000, 0.636966]),

    8: np.array([0.090049, 0.000001, 0.094558, 0.058201, 0.964454, 0.566320, 0.154497, 0.712821]),

}

W9Y = {1:0.0,2:0.5641261834450958,3:-0.006194559125586974,

       4:0.5477069467900759,5:4519.925222262125,6:-0.2780636435928158,

       7:2.2506650591726416,8:9.9417719874239}

W10X = {

    1: np.array([0.098904, 0.300459]), 2: np.array([0.715876, 0.005989]),

    3: np.array([0.196663, 0.981783, 0.000674]),

    4: np.array([0.409992, 0.382531, 0.400832, 0.410710]),

    5: np.array([0.468240, 0.999999, 0.999999, 0.999999]),

    6: np.array([0.525876, 0.321840, 0.718922, 0.725188, 0.164633]),

    7: np.array([0.064058, 0.024164, 0.298611, 0.247300, 0.389429, 0.590442]),

    8: np.array([0.088168, 0.185331, 0.145690, 0.113714, 0.786327, 0.491018, 0.182694, 0.611969]),

}

W10Y = {1:4.0734386728873667e-84,2:0.47964658201500954,3:-0.14286108789054822,

        4:0.4367391324245138,5:4624.456312526239,6:-0.3539795636287327,

        7:2.335166080706118,8:9.9956290531384}



ANCHOR = {1: W2X[1], 2: W5X[2], 3: W9X[3], 4: W4X[4],

          5: W10X[5], 6: W7X[6], 7: W10X[7], 8: W10X[8]}

POLICY = {

    1: dict(kappa=1.8, local=0.08, min_dist=0.025),

    2: dict(kappa=0.5, local=0.015, min_dist=0.006),

    3: dict(kappa=0.8, local=0.05, min_dist=0.02),

    4: dict(kappa=0.65, local=0.03, min_dist=0.015),

    5: dict(kappa=0.35, local=0.03, min_dist=0.01),

    6: dict(kappa=0.7, local=0.04, min_dist=0.02),

    7: dict(kappa=0.55, local=0.04, min_dist=0.018),

    8: dict(kappa=0.55, local=0.04, min_dist=0.03),

}



def fmt(x):

    x = np.clip(np.asarray(x, float).ravel(), 1e-6, 1-1e-6)

    return "-".join(f"{v:.6f}" for v in x)



queries = {}

for k in range(1, 9):

    X0 = np.load(base / f"function_{k}" / "initial_inputs.npy").astype(float)

    y0 = np.load(base / f"function_{k}" / "initial_outputs.npy").astype(float).ravel()

    X = np.vstack([X0, W1X[k], W2X[k], W3X[k], W4X[k], W5X[k], W6X[k], W7X[k], W8X[k], W9X[k], W10X[k]])

    y = np.concatenate([y0, [W1Y[k], W2Y[k], W3Y[k], W4Y[k], W5Y[k], W6Y[k], W7Y[k], W8Y[k], W9Y[k], W10Y[k]]])

    d, pol, rng = DIMS[k], POLICY[k], np.random.default_rng(11000 + k)

    print(f"F{k} n={len(y)} best={y.max():.6g} last={y[-1]:.6g}")

    xsc = StandardScaler().fit(X)

    gp = GaussianProcessRegressor(

        kernel=(ConstantKernel(1.0, (1e-3, 1e3))

                * Matern(np.full(d, 0.3), (0.05, 8.0), nu=2.5)

                + WhiteKernel(1e-5, (1e-12, 1e-1))),

        normalize_y=True, n_restarts_optimizer=2, random_state=k)

    gp.fit(xsc.transform(X), y)

    cand = np.clip(ANCHOR[k] + rng.normal(0, pol["local"], size=(9000, d)), 1e-6, 1-1e-6)

    cand = np.vstack([cand, rng.uniform(1e-6, 1-1e-6, size=(2000, d))])

    mu, std = gp.predict(xsc.transform(cand), return_std=True)

    scores = mu + pol["kappa"] * std

    dist = np.min(np.linalg.norm(cand[:, None, :] - X[None, :, :], axis=2), axis=1)

    scores = np.where(dist < pol["min_dist"], -np.inf, scores)

    q = fmt(cand[int(np.argmax(scores))])

    queries[k] = q

    print("  SUBMIT:", q, "\n")



print("=" * 60)

print("WEEK 11 — PASTE INTO THE PORTAL")

print("=" * 60)

for k, q in queries.items():

    print(f"Function {k}: {q}")

F1 n=20 best=4.58414e-13 last=4.07344e-84
  SUBMIT: 0.969357-0.372109 

F2 n=20 best=0.723385 last=0.479647


C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified upper bound 8.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified upper bound 0.1. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


  SUBMIT: 0.835062-0.241041 

F3 n=25 best=-0.00619456 last=-0.142861
  SUBMIT: 0.554868-0.390496-0.446638 

F4 n=40 best=0.609851 last=0.436739
  SUBMIT: 0.417123-0.387674-0.412121-0.423966 

F5 n=30 best=4624.46 last=4624.46


C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified upper bound 8.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 2 of parameter k1__k2__length_scale is close to the specified upper bound 8.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\gaussian_process\kernels.py:445: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-12. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


  SUBMIT: 0.568914-0.999999-0.999999-0.999999 

F6 n=30 best=-0.162302 last=-0.35398
  SUBMIT: 0.524147-0.469192-0.633198-0.755336-0.078410 

F7 n=40 best=2.33517 last=2.33517


C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 2 of parameter k1__k2__length_scale is close to the specified upper bound 8.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


  SUBMIT: 0.062334-0.087819-0.293457-0.218299-0.403296-0.572137 

F8 n=50 best=9.99563 last=9.99563
  SUBMIT: 0.082637-0.218670-0.144065-0.176029-0.838483-0.487831-0.199565-0.635253 

WEEK 11 — PASTE INTO THE PORTAL
Function 1: 0.969357-0.372109
Function 2: 0.835062-0.241041
Function 3: 0.554868-0.390496-0.446638
Function 4: 0.417123-0.387674-0.412121-0.423966
Function 5: 0.568914-0.999999-0.999999-0.999999
Function 6: 0.524147-0.469192-0.633198-0.755336-0.078410
Function 7: 0.062334-0.087819-0.293457-0.218299-0.403296-0.572137
Function 8: 0.082637-0.218670-0.144065-0.176029-0.838483-0.487831-0.199565-0.635253


C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 3 of parameter k1__k2__length_scale is close to the specified upper bound 8.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 4 of parameter k1__k2__length_scale is close to the specified upper bound 8.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 5 of parameter k1__k2__length_scale is close to the specified upper bound 8.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
C:\Users\R

In [1]:
import numpy as np

from pathlib import Path

from sklearn.gaussian_process import GaussianProcessRegressor

from sklearn.gaussian_process.kernels import ConstantKernel, Matern, WhiteKernel

from sklearn.preprocessing import StandardScaler



base = Path(r"C:\Users\Ramnath Lakshmanan\Downloads\Initial_data_points_starter (2)\initial_data")

DIMS = {1: 2, 2: 2, 3: 3, 4: 4, 5: 4, 6: 5, 7: 6, 8: 8}



W1X = {

    1: np.array([0.973822, 0.110674]), 2: np.array([0.711216, 0.383626]),

    3: np.array([0.628013, 0.359224, 0.539509]),

    4: np.array([0.367191, 0.464182, 0.391040, 0.448820]),

    5: np.array([0.245752, 0.932264, 0.962162, 0.965058]),

    6: np.array([0.446820, 0.321047, 0.436584, 0.762321, 0.003307]),

    7: np.array([0.086332, 0.371883, 0.229157, 0.106524, 0.394874, 0.689739]),

    8: np.array([0.054501, 0.081016, 0.320114, 0.398440, 0.621808, 0.649205, 0.338643, 0.601487]),

}

W1Y = {1:-1.1814831817160104e-270,2:0.6525848308450871,3:-0.01927280144331694,

       4:-0.9174251874710708,5:2830.5996301272953,6:-0.41995556157505487,

       7:1.5145555220314069,8:9.7443661466821}

W2X = {

    1: np.array([0.520875, 0.334357]), 2: np.array([0.701530, 0.047056]),

    3: np.array([0.879583, 0.006977, 0.001530]),

    4: np.array([0.421199, 0.400087, 0.337912, 0.436790]),

    5: np.array([0.116598, 0.999999, 0.999999, 0.999999]),

    6: np.array([0.425618, 0.196929, 0.846445, 0.972258, 0.141083]),

    7: np.array([0.000001, 0.312636, 0.151505, 0.026906, 0.361543, 0.731254]),

    8: np.array([0.102526, 0.102909, 0.008115, 0.245476, 0.999999, 0.523852, 0.248637, 0.499859]),

}

W2Y = {1:4.584141793749554e-13,2:0.6685300747584026,3:-0.18176416202017948,

       4:0.2852616086635966,5:4441.586450664394,6:-0.5975592068881134,

       7:1.0217094038661905,8:9.9177834946854}

W3X = {

    1: np.array([0.605824, 0.805874]), 2: np.array([0.719374, 0.085613]),

    3: np.array([0.040729, 0.915411, 0.504944]),

    4: np.array([0.439246, 0.393165, 0.273345, 0.439606]),

    5: np.array([0.000001, 0.999999, 0.999999, 0.999999]),

    6: np.array([0.508487, 0.271802, 0.367910, 0.999999, 0.000001]),

    7: np.array([0.078771, 0.282330, 0.246212, 0.164149, 0.401057, 0.641158]),

    8: np.array([0.179705, 0.000001, 0.204942, 0.149746, 0.990645, 0.489105, 0.181870, 0.440792]),

}

W3Y = {1:5.221216935504337e-23,2:0.6685229247231236,3:-0.030942339134816123,

       4:-1.2975118969582797,5:4440.480873839292,6:-0.7811647047830718,

       7:1.9868235528281357,8:9.92646197977771}

W4X = {

    1: np.array([0.906435, 0.524773]), 2: np.array([0.711541, 0.000001]),

    3: np.array([0.901391, 0.100657, 0.501946]),

    4: np.array([0.404491, 0.411487, 0.374128, 0.419096]),

    5: np.array([0.070905, 0.999999, 0.999999, 0.812090]),

    6: np.array([0.423685, 0.314205, 0.861819, 0.607621, 0.009157]),

    7: np.array([0.053889, 0.170531, 0.230571, 0.223537, 0.393017, 0.615148]),

    8: np.array([0.000001, 0.000001, 0.058234, 0.160975, 0.905106, 0.422655, 0.072201, 0.456271]),

}

W4Y = {1:-9.171561473040102e-63,2:0.718243930604028,3:-0.05350002457722131,

       4:0.6098506102572618,5:2690.3948413647527,6:-0.6049887641373398,

       7:2.2735115338126204,8:9.8956923171149}

W5X = {

    1: np.array([0.276310, 0.463966]), 2: np.array([0.711701, 0.012157]),

    3: np.array([0.812590, 0.971252, 0.407756]),

    4: np.array([0.449239, 0.320819, 0.430682, 0.426303]),

    5: np.array([0.029536, 0.999999, 0.999999, 0.999999]),

    6: np.array([0.440177, 0.307928, 0.516852, 0.718082, 0.163715]),

    7: np.array([0.000001, 0.079068, 0.203789, 0.237496, 0.371176, 0.578991]),

    8: np.array([0.116622, 0.061321, 0.145257, 0.029806, 0.891611, 0.487268, 0.132540, 0.643656]),

}

W5Y = {1:-1.4588117677798244e-17,2:0.7233854034475573,3:-0.0348256634794126,

       4:-0.3587979590239736,5:4440.508119101403,6:-0.3636516136843698,

       7:2.113146751542243,8:9.9627878480899}

W6X = {

    1: np.array([0.147012, 0.207382]), 2: np.array([0.711320, 0.022518]),

    3: np.array([0.042548, 0.018175, 0.535544]),

    4: np.array([0.417683, 0.392791, 0.389865, 0.415352]),

    5: np.array([0.072146, 0.999999, 0.999999, 0.999999]),

    6: np.array([0.483442, 0.406320, 0.551572, 0.728395, 0.141301]),

    7: np.array([0.002952, 0.137104, 0.221381, 0.265099, 0.399907, 0.616118]),

    8: np.array([0.024400, 0.068670, 0.123985, 0.088530, 0.958126, 0.519233, 0.251301, 0.734513]),

}

W6Y = {1:7.513182419081889e-87,2:0.5266505044433385,3:-0.10301859579872755,

       4:0.45948056013104877,5:4440.724001387989,6:-0.29074974600067394,

       7:2.1465849895947136,8:9.9581228253791}

W7X = {

    1: np.array([0.623326, 0.011857]), 2: np.array([0.688509, 0.006758]),

    3: np.array([0.973709, 0.001813, 0.792312]),

    4: np.array([0.411661, 0.384885, 0.399073, 0.428315]),

    5: np.array([0.206499, 0.999999, 0.999999, 0.999999]),

    6: np.array([0.528613, 0.408984, 0.624132, 0.735857, 0.121553]),

    7: np.array([0.076466, 0.107136, 0.246213, 0.221705, 0.391060, 0.600276]),

    8: np.array([0.123965, 0.028657, 0.123383, 0.188126, 0.900411, 0.481502, 0.091022, 0.738327]),

}

W7Y = {1:2.1604429724905267e-146,2:0.5854363115719756,3:-0.1366850708077359,

       4:0.43813490537661126,5:4448.754616848498,6:-0.1623020733630079,

       7:2.3287657825776096,8:9.9514930826326}

W8X = {

    1: np.array([0.724072, 0.949508]), 2: np.array([0.728329, 0.006445]),

    3: np.array([0.451949, 0.969232, 0.458722]),

    4: np.array([0.404607, 0.397025, 0.403666, 0.417850]),

    5: np.array([0.301402, 0.999999, 0.999999, 0.999999]),

    6: np.array([0.538213, 0.415090, 0.660675, 0.743991, 0.110951]),

    7: np.array([0.119224, 0.110793, 0.167022, 0.243798, 0.404717, 0.589881]),

    8: np.array([0.112099, 0.000001, 0.112888, 0.055898, 0.872203, 0.396747, 0.171766, 0.681514]),

}

W8Y = {1:-1.7695749378252557e-78,2:0.5691554534436665,3:-0.03431158030376151,

       4:0.4792797063547316,5:4474.337323581212,6:-0.18658854384879187,

       7:2.186293628123943,8:9.9519472910159}

W9X = {

    1: np.array([0.032731, 0.999991]), 2: np.array([0.705749, 0.006184]),

    3: np.array([0.852022, 0.624763, 0.509852]),

    4: np.array([0.401861, 0.377528, 0.411745, 0.423788]),

    5: np.array([0.376936, 0.999999, 0.999999, 0.999999]),

    6: np.array([0.558753, 0.355800, 0.638855, 0.725691, 0.108270]),

    7: np.array([0.063449, 0.102659, 0.338015, 0.219698, 0.430000, 0.636966]),

    8: np.array([0.090049, 0.000001, 0.094558, 0.058201, 0.964454, 0.566320, 0.154497, 0.712821]),

}

W9Y = {1:0.0,2:0.5641261834450958,3:-0.006194559125586974,

       4:0.5477069467900759,5:4519.925222262125,6:-0.2780636435928158,

       7:2.2506650591726416,8:9.9417719874239}

W10X = {

    1: np.array([0.098904, 0.300459]), 2: np.array([0.715876, 0.005989]),

    3: np.array([0.196663, 0.981783, 0.000674]),

    4: np.array([0.409992, 0.382531, 0.400832, 0.410710]),

    5: np.array([0.468240, 0.999999, 0.999999, 0.999999]),

    6: np.array([0.525876, 0.321840, 0.718922, 0.725188, 0.164633]),

    7: np.array([0.064058, 0.024164, 0.298611, 0.247300, 0.389429, 0.590442]),

    8: np.array([0.088168, 0.185331, 0.145690, 0.113714, 0.786327, 0.491018, 0.182694, 0.611969]),

}

W10Y = {1:4.0734386728873667e-84,2:0.47964658201500954,3:-0.14286108789054822,

        4:0.4367391324245138,5:4624.456312526239,6:-0.3539795636287327,

        7:2.335166080706118,8:9.9956290531384}

W11X = {

    1: np.array([0.969357, 0.372109]), 2: np.array([0.835062, 0.241041]),

    3: np.array([0.554868, 0.390496, 0.446638]),

    4: np.array([0.417123, 0.387674, 0.412121, 0.423966]),

    5: np.array([0.568914, 0.999999, 0.999999, 0.999999]),

    6: np.array([0.524147, 0.469192, 0.633198, 0.755336, 0.078410]),

    7: np.array([0.062334, 0.087819, 0.293457, 0.218299, 0.403296, 0.572137]),

    8: np.array([0.082637, 0.218670, 0.144065, 0.176029, 0.838483, 0.487831, 0.199565, 0.635253]),

}

W11Y = {1:-2.553692637618622e-127,2:0.27850470496564816,3:-0.005668180864493751,

        4:0.6122063318422168,5:4836.285945017753,6:-0.2318604765164367,

        7:2.2134202915451002,8:9.9923972909896}



ANCHOR = {1: W2X[1], 2: W5X[2], 3: W11X[3], 4: W11X[4],

          5: W11X[5], 6: W7X[6], 7: W10X[7], 8: W10X[8]}

POLICY = {

    1: dict(kappa=1.8, local=0.08, min_dist=0.025),

    2: dict(kappa=0.45, local=0.012, min_dist=0.005),

    3: dict(kappa=0.7, local=0.04, min_dist=0.015),

    4: dict(kappa=0.55, local=0.025, min_dist=0.012),

    5: dict(kappa=0.3, local=0.03, min_dist=0.01),

    6: dict(kappa=0.7, local=0.04, min_dist=0.02),

    7: dict(kappa=0.55, local=0.035, min_dist=0.015),

    8: dict(kappa=0.5, local=0.035, min_dist=0.025),

}



def fmt(x):

    x = np.clip(np.asarray(x, float).ravel(), 1e-6, 1-1e-6)

    return "-".join(f"{v:.6f}" for v in x)



queries = {}

for k in range(1, 9):

    X0 = np.load(base / f"function_{k}" / "initial_inputs.npy").astype(float)

    y0 = np.load(base / f"function_{k}" / "initial_outputs.npy").astype(float).ravel()

    X = np.vstack([X0, W1X[k], W2X[k], W3X[k], W4X[k], W5X[k], W6X[k],

                   W7X[k], W8X[k], W9X[k], W10X[k], W11X[k]])

    y = np.concatenate([y0, [W1Y[k], W2Y[k], W3Y[k], W4Y[k], W5Y[k], W6Y[k],

                            W7Y[k], W8Y[k], W9Y[k], W10Y[k], W11Y[k]]])

    d, pol, rng = DIMS[k], POLICY[k], np.random.default_rng(12000 + k)

    print(f"F{k} n={len(y)} best={y.max():.6g} last={y[-1]:.6g}")

    xsc = StandardScaler().fit(X)

    gp = GaussianProcessRegressor(

        kernel=(ConstantKernel(1.0, (1e-3, 1e3))

                * Matern(np.full(d, 0.3), (0.05, 8.0), nu=2.5)

                + WhiteKernel(1e-5, (1e-12, 1e-1))),

        normalize_y=True, n_restarts_optimizer=2, random_state=k)

    gp.fit(xsc.transform(X), y)

    cand = np.clip(ANCHOR[k] + rng.normal(0, pol["local"], size=(9000, d)), 1e-6, 1-1e-6)

    cand = np.vstack([cand, rng.uniform(1e-6, 1-1e-6, size=(1500, d))])

    mu, std = gp.predict(xsc.transform(cand), return_std=True)

    scores = mu + pol["kappa"] * std

    dist = np.min(np.linalg.norm(cand[:, None, :] - X[None, :, :], axis=2), axis=1)

    scores = np.where(dist < pol["min_dist"], -np.inf, scores)

    q = fmt(cand[int(np.argmax(scores))])

    queries[k] = q

    print("  SUBMIT:", q, "\n")



print("=" * 60)

print("WEEK 12 — PASTE INTO THE PORTAL")

print("=" * 60)

for k, q in queries.items():

    print(f"Function {k}: {q}")

F1 n=21 best=4.58414e-13 last=-2.55369e-127
  SUBMIT: 0.944128-0.046586 

F2 n=21 best=0.723385 last=0.278505


C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified upper bound 8.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 8.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


  SUBMIT: 0.708728-0.412267 

F3 n=26 best=-0.00566818 last=-0.00566818
  SUBMIT: 0.765816-0.517926-0.458452 

F4 n=41 best=0.612206 last=0.612206
  SUBMIT: 0.408703-0.387687-0.415617-0.413897 

F5 n=31 best=4836.29 last=4836.29
  SUBMIT: 0.672320-0.999999-0.999999-0.999999 

F6 n=31 best=-0.162302 last=-0.23186


C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\gaussian_process\_gpr.py:667: ConvergenceWarning: lbfgs failed to converge after 19 iteration(s) (status=2):
ABNORMAL: 

You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  _check_optimize_result("lbfgs", opt_res)
C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified upper bound 8.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 2 of parameter k1__k2__length_scale is close to the specified upper bound 8.0. Increasing the bound and calling fit again may find a better value.
  wa

  SUBMIT: 0.531645-0.456020-0.626602-0.743372-0.083872 

F7 n=41 best=2.33517 last=2.21342


C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 2 of parameter k1__k2__length_scale is close to the specified upper bound 8.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


  SUBMIT: 0.076002-0.000001-0.298339-0.228309-0.362561-0.630383 

F8 n=51 best=9.99563 last=9.9924
  SUBMIT: 0.052614-0.125800-0.139908-0.158215-0.772790-0.535395-0.181558-0.674756 

WEEK 12 — PASTE INTO THE PORTAL
Function 1: 0.944128-0.046586
Function 2: 0.708728-0.412267
Function 3: 0.765816-0.517926-0.458452
Function 4: 0.408703-0.387687-0.415617-0.413897
Function 5: 0.672320-0.999999-0.999999-0.999999
Function 6: 0.531645-0.456020-0.626602-0.743372-0.083872
Function 7: 0.076002-0.000001-0.298339-0.228309-0.362561-0.630383
Function 8: 0.052614-0.125800-0.139908-0.158215-0.772790-0.535395-0.181558-0.674756


C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 3 of parameter k1__k2__length_scale is close to the specified upper bound 8.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 4 of parameter k1__k2__length_scale is close to the specified upper bound 8.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 5 of parameter k1__k2__length_scale is close to the specified upper bound 8.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
C:\Users\R

In [1]:
import numpy as np

from pathlib import Path

from sklearn.gaussian_process import GaussianProcessRegressor

from sklearn.gaussian_process.kernels import ConstantKernel, Matern, WhiteKernel

from sklearn.preprocessing import StandardScaler



base = Path(r"C:\Users\Ramnath Lakshmanan\Downloads\Initial_data_points_starter (2)\initial_data")

DIMS = {1: 2, 2: 2, 3: 3, 4: 4, 5: 4, 6: 5, 7: 6, 8: 8}



W1X = {

    1: np.array([0.973822, 0.110674]), 2: np.array([0.711216, 0.383626]),

    3: np.array([0.628013, 0.359224, 0.539509]),

    4: np.array([0.367191, 0.464182, 0.391040, 0.448820]),

    5: np.array([0.245752, 0.932264, 0.962162, 0.965058]),

    6: np.array([0.446820, 0.321047, 0.436584, 0.762321, 0.003307]),

    7: np.array([0.086332, 0.371883, 0.229157, 0.106524, 0.394874, 0.689739]),

    8: np.array([0.054501, 0.081016, 0.320114, 0.398440, 0.621808, 0.649205, 0.338643, 0.601487]),

}

W1Y = {1:-1.1814831817160104e-270,2:0.6525848308450871,3:-0.01927280144331694,

       4:-0.9174251874710708,5:2830.5996301272953,6:-0.41995556157505487,

       7:1.5145555220314069,8:9.7443661466821}

W2X = {

    1: np.array([0.520875, 0.334357]), 2: np.array([0.701530, 0.047056]),

    3: np.array([0.879583, 0.006977, 0.001530]),

    4: np.array([0.421199, 0.400087, 0.337912, 0.436790]),

    5: np.array([0.116598, 0.999999, 0.999999, 0.999999]),

    6: np.array([0.425618, 0.196929, 0.846445, 0.972258, 0.141083]),

    7: np.array([0.000001, 0.312636, 0.151505, 0.026906, 0.361543, 0.731254]),

    8: np.array([0.102526, 0.102909, 0.008115, 0.245476, 0.999999, 0.523852, 0.248637, 0.499859]),

}

W2Y = {1:4.584141793749554e-13,2:0.6685300747584026,3:-0.18176416202017948,

       4:0.2852616086635966,5:4441.586450664394,6:-0.5975592068881134,

       7:1.0217094038661905,8:9.9177834946854}

W3X = {

    1: np.array([0.605824, 0.805874]), 2: np.array([0.719374, 0.085613]),

    3: np.array([0.040729, 0.915411, 0.504944]),

    4: np.array([0.439246, 0.393165, 0.273345, 0.439606]),

    5: np.array([0.000001, 0.999999, 0.999999, 0.999999]),

    6: np.array([0.508487, 0.271802, 0.367910, 0.999999, 0.000001]),

    7: np.array([0.078771, 0.282330, 0.246212, 0.164149, 0.401057, 0.641158]),

    8: np.array([0.179705, 0.000001, 0.204942, 0.149746, 0.990645, 0.489105, 0.181870, 0.440792]),

}

W3Y = {1:5.221216935504337e-23,2:0.6685229247231236,3:-0.030942339134816123,

       4:-1.2975118969582797,5:4440.480873839292,6:-0.7811647047830718,

       7:1.9868235528281357,8:9.92646197977771}

W4X = {

    1: np.array([0.906435, 0.524773]), 2: np.array([0.711541, 0.000001]),

    3: np.array([0.901391, 0.100657, 0.501946]),

    4: np.array([0.404491, 0.411487, 0.374128, 0.419096]),

    5: np.array([0.070905, 0.999999, 0.999999, 0.812090]),

    6: np.array([0.423685, 0.314205, 0.861819, 0.607621, 0.009157]),

    7: np.array([0.053889, 0.170531, 0.230571, 0.223537, 0.393017, 0.615148]),

    8: np.array([0.000001, 0.000001, 0.058234, 0.160975, 0.905106, 0.422655, 0.072201, 0.456271]),

}

W4Y = {1:-9.171561473040102e-63,2:0.718243930604028,3:-0.05350002457722131,

       4:0.6098506102572618,5:2690.3948413647527,6:-0.6049887641373398,

       7:2.2735115338126204,8:9.8956923171149}

W5X = {

    1: np.array([0.276310, 0.463966]), 2: np.array([0.711701, 0.012157]),

    3: np.array([0.812590, 0.971252, 0.407756]),

    4: np.array([0.449239, 0.320819, 0.430682, 0.426303]),

    5: np.array([0.029536, 0.999999, 0.999999, 0.999999]),

    6: np.array([0.440177, 0.307928, 0.516852, 0.718082, 0.163715]),

    7: np.array([0.000001, 0.079068, 0.203789, 0.237496, 0.371176, 0.578991]),

    8: np.array([0.116622, 0.061321, 0.145257, 0.029806, 0.891611, 0.487268, 0.132540, 0.643656]),

}

W5Y = {1:-1.4588117677798244e-17,2:0.7233854034475573,3:-0.0348256634794126,

       4:-0.3587979590239736,5:4440.508119101403,6:-0.3636516136843698,

       7:2.113146751542243,8:9.9627878480899}

W6X = {

    1: np.array([0.147012, 0.207382]), 2: np.array([0.711320, 0.022518]),

    3: np.array([0.042548, 0.018175, 0.535544]),

    4: np.array([0.417683, 0.392791, 0.389865, 0.415352]),

    5: np.array([0.072146, 0.999999, 0.999999, 0.999999]),

    6: np.array([0.483442, 0.406320, 0.551572, 0.728395, 0.141301]),

    7: np.array([0.002952, 0.137104, 0.221381, 0.265099, 0.399907, 0.616118]),

    8: np.array([0.024400, 0.068670, 0.123985, 0.088530, 0.958126, 0.519233, 0.251301, 0.734513]),

}

W6Y = {1:7.513182419081889e-87,2:0.5266505044433385,3:-0.10301859579872755,

       4:0.45948056013104877,5:4440.724001387989,6:-0.29074974600067394,

       7:2.1465849895947136,8:9.9581228253791}

W7X = {

    1: np.array([0.623326, 0.011857]), 2: np.array([0.688509, 0.006758]),

    3: np.array([0.973709, 0.001813, 0.792312]),

    4: np.array([0.411661, 0.384885, 0.399073, 0.428315]),

    5: np.array([0.206499, 0.999999, 0.999999, 0.999999]),

    6: np.array([0.528613, 0.408984, 0.624132, 0.735857, 0.121553]),

    7: np.array([0.076466, 0.107136, 0.246213, 0.221705, 0.391060, 0.600276]),

    8: np.array([0.123965, 0.028657, 0.123383, 0.188126, 0.900411, 0.481502, 0.091022, 0.738327]),

}

W7Y = {1:2.1604429724905267e-146,2:0.5854363115719756,3:-0.1366850708077359,

       4:0.43813490537661126,5:4448.754616848498,6:-0.1623020733630079,

       7:2.3287657825776096,8:9.9514930826326}

W8X = {

    1: np.array([0.724072, 0.949508]), 2: np.array([0.728329, 0.006445]),

    3: np.array([0.451949, 0.969232, 0.458722]),

    4: np.array([0.404607, 0.397025, 0.403666, 0.417850]),

    5: np.array([0.301402, 0.999999, 0.999999, 0.999999]),

    6: np.array([0.538213, 0.415090, 0.660675, 0.743991, 0.110951]),

    7: np.array([0.119224, 0.110793, 0.167022, 0.243798, 0.404717, 0.589881]),

    8: np.array([0.112099, 0.000001, 0.112888, 0.055898, 0.872203, 0.396747, 0.171766, 0.681514]),

}

W8Y = {1:-1.7695749378252557e-78,2:0.5691554534436665,3:-0.03431158030376151,

       4:0.4792797063547316,5:4474.337323581212,6:-0.18658854384879187,

       7:2.186293628123943,8:9.9519472910159}

W9X = {

    1: np.array([0.032731, 0.999991]), 2: np.array([0.705749, 0.006184]),

    3: np.array([0.852022, 0.624763, 0.509852]),

    4: np.array([0.401861, 0.377528, 0.411745, 0.423788]),

    5: np.array([0.376936, 0.999999, 0.999999, 0.999999]),

    6: np.array([0.558753, 0.355800, 0.638855, 0.725691, 0.108270]),

    7: np.array([0.063449, 0.102659, 0.338015, 0.219698, 0.430000, 0.636966]),

    8: np.array([0.090049, 0.000001, 0.094558, 0.058201, 0.964454, 0.566320, 0.154497, 0.712821]),

}

W9Y = {1:0.0,2:0.5641261834450958,3:-0.006194559125586974,

       4:0.5477069467900759,5:4519.925222262125,6:-0.2780636435928158,

       7:2.2506650591726416,8:9.9417719874239}

W10X = {

    1: np.array([0.098904, 0.300459]), 2: np.array([0.715876, 0.005989]),

    3: np.array([0.196663, 0.981783, 0.000674]),

    4: np.array([0.409992, 0.382531, 0.400832, 0.410710]),

    5: np.array([0.468240, 0.999999, 0.999999, 0.999999]),

    6: np.array([0.525876, 0.321840, 0.718922, 0.725188, 0.164633]),

    7: np.array([0.064058, 0.024164, 0.298611, 0.247300, 0.389429, 0.590442]),

    8: np.array([0.088168, 0.185331, 0.145690, 0.113714, 0.786327, 0.491018, 0.182694, 0.611969]),

}

W10Y = {1:4.0734386728873667e-84,2:0.47964658201500954,3:-0.14286108789054822,

        4:0.4367391324245138,5:4624.456312526239,6:-0.3539795636287327,

        7:2.335166080706118,8:9.9956290531384}

W11X = {

    1: np.array([0.969357, 0.372109]), 2: np.array([0.835062, 0.241041]),

    3: np.array([0.554868, 0.390496, 0.446638]),

    4: np.array([0.417123, 0.387674, 0.412121, 0.423966]),

    5: np.array([0.568914, 0.999999, 0.999999, 0.999999]),

    6: np.array([0.524147, 0.469192, 0.633198, 0.755336, 0.078410]),

    7: np.array([0.062334, 0.087819, 0.293457, 0.218299, 0.403296, 0.572137]),

    8: np.array([0.082637, 0.218670, 0.144065, 0.176029, 0.838483, 0.487831, 0.199565, 0.635253]),

}

W11Y = {1:-2.553692637618622e-127,2:0.27850470496564816,3:-0.005668180864493751,

        4:0.6122063318422168,5:4836.285945017753,6:-0.2318604765164367,

        7:2.2134202915451002,8:9.9923972909896}

W12X = {

    1: np.array([0.944128, 0.046586]), 2: np.array([0.708728, 0.412267]),

    3: np.array([0.765816, 0.517926, 0.458452]),

    4: np.array([0.408703, 0.387687, 0.415617, 0.413897]),

    5: np.array([0.672320, 0.999999, 0.999999, 0.999999]),

    6: np.array([0.531645, 0.456020, 0.626602, 0.743372, 0.083872]),

    7: np.array([0.076002, 0.000001, 0.298339, 0.228309, 0.362561, 0.630383]),

    8: np.array([0.052614, 0.125800, 0.139908, 0.158215, 0.772790, 0.535395, 0.181558, 0.674756]),

}

W12Y = {1:1.1795160350959129e-288,2:0.4795534408225325,3:-0.008404439835138612,

        4:0.5802967698297725,5:5214.085125635222,6:-0.3099738378322989,

        7:2.48625284999962604,8:9.9916994436344}



ANCHOR = {1: W2X[1], 2: W5X[2], 3: W11X[3], 4: W11X[4],

          5: W12X[5], 6: W7X[6], 7: W12X[7], 8: W10X[8]}

POLICY = {

    1: dict(kappa=1.6, local=0.07, min_dist=0.02),

    2: dict(kappa=0.4, local=0.012, min_dist=0.005),

    3: dict(kappa=0.6, local=0.035, min_dist=0.015),

    4: dict(kappa=0.5, local=0.02, min_dist=0.01),

    5: dict(kappa=0.25, local=0.03, min_dist=0.01),

    6: dict(kappa=0.6, local=0.035, min_dist=0.018),

    7: dict(kappa=0.45, local=0.03, min_dist=0.012),

    8: dict(kappa=0.45, local=0.03, min_dist=0.02),

}



def fmt(x):

    x = np.clip(np.asarray(x, float).ravel(), 1e-6, 1-1e-6)

    return "-".join(f"{v:.6f}" for v in x)



queries = {}

for k in range(1, 9):

    X0 = np.load(base / f"function_{k}" / "initial_inputs.npy").astype(float)

    y0 = np.load(base / f"function_{k}" / "initial_outputs.npy").astype(float).ravel()

    X = np.vstack([X0, W1X[k], W2X[k], W3X[k], W4X[k], W5X[k], W6X[k],

                   W7X[k], W8X[k], W9X[k], W10X[k], W11X[k], W12X[k]])

    y = np.concatenate([y0, [W1Y[k], W2Y[k], W3Y[k], W4Y[k], W5Y[k], W6Y[k],

                            W7Y[k], W8Y[k], W9Y[k], W10Y[k], W11Y[k], W12Y[k]]])

    d, pol, rng = DIMS[k], POLICY[k], np.random.default_rng(13000 + k)

    print(f"F{k} n={len(y)} best={y.max():.6g} last={y[-1]:.6g}")

    xsc = StandardScaler().fit(X)

    gp = GaussianProcessRegressor(

        kernel=(ConstantKernel(1.0, (1e-3, 1e3))

                * Matern(np.full(d, 0.3), (0.05, 8.0), nu=2.5)

                + WhiteKernel(1e-5, (1e-12, 1e-1))),

        normalize_y=True, n_restarts_optimizer=2, random_state=k)

    gp.fit(xsc.transform(X), y)

    cand = np.clip(ANCHOR[k] + rng.normal(0, pol["local"], size=(9000, d)), 1e-6, 1-1e-6)

    cand = np.vstack([cand, rng.uniform(1e-6, 1-1e-6, size=(1200, d))])

    mu, std = gp.predict(xsc.transform(cand), return_std=True)

    scores = mu + pol["kappa"] * std

    dist = np.min(np.linalg.norm(cand[:, None, :] - X[None, :, :], axis=2), axis=1)

    scores = np.where(dist < pol["min_dist"], -np.inf, scores)

    q = fmt(cand[int(np.argmax(scores))])

    queries[k] = q

    print("  SUBMIT:", q, "\n")



print("=" * 60)

print("FINAL WEEK — PASTE THE CELL OUTPUT INTO THE PORTAL")

print("=" * 60)

for k, q in queries.items():

    print(f"Function {k}: {q}")

F1 n=22 best=4.58414e-13 last=1.17952e-288
  SUBMIT: 0.063374-0.552641 

F2 n=22 best=0.723385 last=0.479553


C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified upper bound 8.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 8.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified upper bound 0.1. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


  SUBMIT: 0.704815-0.000001 

F3 n=27 best=-0.00566818 last=-0.00840444
  SUBMIT: 0.987911-0.199775-0.408049 

F4 n=42 best=0.612206 last=0.580297


C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified upper bound 8.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


  SUBMIT: 0.405339-0.384696-0.406659-0.417614 

F5 n=32 best=5214.09 last=5214.09


C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 2 of parameter k1__k2__length_scale is close to the specified upper bound 8.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\gaussian_process\kernels.py:445: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-12. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


  SUBMIT: 0.770715-0.999999-0.999999-0.999999 

F6 n=32 best=-0.162302 last=-0.309974
  SUBMIT: 0.518112-0.444066-0.631339-0.755503-0.127365 

F7 n=42 best=2.48625 last=2.48625


C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 2 of parameter k1__k2__length_scale is close to the specified upper bound 8.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


  SUBMIT: 0.090784-0.000001-0.340448-0.237765-0.349169-0.647671 

F8 n=52 best=9.99563 last=9.9917
  SUBMIT: 0.120179-0.164217-0.152878-0.157702-0.799478-0.509155-0.202802-0.621526 

FINAL WEEK — PASTE THE CELL OUTPUT INTO THE PORTAL
Function 1: 0.063374-0.552641
Function 2: 0.704815-0.000001
Function 3: 0.987911-0.199775-0.408049
Function 4: 0.405339-0.384696-0.406659-0.417614
Function 5: 0.770715-0.999999-0.999999-0.999999
Function 6: 0.518112-0.444066-0.631339-0.755503-0.127365
Function 7: 0.090784-0.000001-0.340448-0.237765-0.349169-0.647671
Function 8: 0.120179-0.164217-0.152878-0.157702-0.799478-0.509155-0.202802-0.621526


C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 3 of parameter k1__k2__length_scale is close to the specified upper bound 8.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 4 of parameter k1__k2__length_scale is close to the specified upper bound 8.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
C:\Users\Ramnath Lakshmanan\anaconda3\New folder\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 5 of parameter k1__k2__length_scale is close to the specified upper bound 8.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
C:\Users\R